In [ ]:
from pathlib import Path
import pandas as pd
import scipy.io

# ---------- user inputs ----------
samples=['DRG_0199_01','DRG_0198_01','DRG_0199_01','DRG_1215_02','DRG_2324_02','DRG_2902_02','DRG_5467_02','DRG_1409_01']

base = Path("/home/913/dk4874/scratch/gdata/scDaisychain_paper/human/processed_data")

singlecell_base = base / "singlecell/transcript_expression_matrices"
bedmethyl_base = base / "bedmethyl"

gtf_path = Path(
    "/home/913/dk4874/scratch/X_Inactivation/raw_data/genome/"
    "homo_sapiens_cellranger/genes/genes.gtf"
)

escapee_base = Path(
    "/home/913/dk4874/scratch/forRob_daisychain/output/escapee_classification"
)

outdir = Path("methylation_top_transcript_merged")
outdir.mkdir(exist_ok=True)

# ---------- columns ----------
bed_cols = [
    "chrom",
    "start position",
    "end position",
    "modified base code",
    "score",
    "strand",
    "start position (compat)",
    "end position (compat)",
    "color",
    "Nvalid_cov",
    "fraction modified",
    "Nmod",
    "Ncanonical",
    "Nother_mod",
    "Ndelete",
    "Nfail",
    "Ndiff",
    "Nnocall",
    "CPG_feature",
    "gene",
]

bed_dtype = {
    "chrom": "string",
    "start position": "int64",
    "end position": "int64",
    "modified base code": "string",
    "score": "Int64",
    "strand": "string",
    "start position (compat)": "Int64",
    "end position (compat)": "Int64",
    "color": "string",
    "Nvalid_cov": "Int64",
    "fraction modified": "float64",
    "Nmod": "Int64",
    "Ncanonical": "Int64",
    "Nother_mod": "Int64",
    "Ndelete": "Int64",
    "Nfail": "Int64",
    "Ndiff": "Int64",
    "Nnocall": "Int64",
    "CPG_feature": "string",
    "gene": "string",
}

# ---------- parse GTF once ----------
def parse_attr(attr, key):
    target = key + ' "'
    if target not in attr:
        return None
    return attr.split(target, 1)[1].split('"', 1)[0]

gtf_rows = []
with open(gtf_path) as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9 or fields[2] != "transcript":
            continue

        attr = fields[8]
        gtf_rows.append({
            "transcript_id": parse_attr(attr, "transcript_id"),
            "gene_id": parse_attr(attr, "gene_id"),
            "gene_name": parse_attr(attr, "gene_name"),
            "chrom_tx": fields[0],
            "tx_start": int(fields[3]),
            "tx_end": int(fields[4]),
            "tx_strand": fields[6],
        })

gtf_tx = (
    pd.DataFrame(gtf_rows)
    .dropna(subset=["transcript_id"])
    .drop_duplicates("transcript_id")
)

# ---------- functions ----------
def get_top_tx_per_gene(sample):
    mtx_dir = singlecell_base / sample

    features = pd.read_csv(
        mtx_dir / "features.tsv.gz",
        sep="\t",
        header=None,
        compression="gzip",
    )
    features.columns = [f"col{i}" for i in range(features.shape[1])]

    X = scipy.io.mmread(mtx_dir / "matrix.mtx.gz").tocsr()

    tx = features.copy()
    tx["count"] = X.sum(axis=1).A1

    # col1 = ENST transcript ID
    tx = tx.rename(columns={"col1": "transcript_id"})

    tx_annot = tx.merge(gtf_tx, on="transcript_id", how="left")

    top_tx_per_gene = (
        tx_annot.dropna(subset=["gene_name"])
        .sort_values(["gene_name", "count"], ascending=[True, False])
        .drop_duplicates("gene_name", keep="first")
        [[
            "gene_name",
            "gene_id",
            "transcript_id",
            "count",
            "chrom_tx",
            "tx_start",
            "tx_end",
            "tx_strand",
        ]]
        .rename(columns={"gene_name": "gene_symbol", "count": "tx_count"})
        .reset_index(drop=True)
    )

    return top_tx_per_gene


def split_and_filter_bed_to_top_tx(df_m, top_tx):
    top_tx_set = set(top_tx["transcript_id"].astype(str))
    top_gene_set = set(top_tx["gene_symbol"].astype(str))

    df_m = df_m.copy()
    df_m["gene_raw"] = df_m["gene"]

    df_expl = (
        df_m.assign(gene_entry=df_m["gene"].fillna(".").astype(str).str.split(","))
        .explode("gene_entry")
        .copy()
    )

    df_expl["gene_entry"] = df_expl["gene_entry"].astype(str)
    df_expl = df_expl[df_expl["gene_entry"].ne(".")].copy()

    parts = df_expl["gene_entry"].str.split("|", n=2, expand=True)

    df_expl["gene_symbol"] = parts[0]

    if parts.shape[1] > 1:
        df_expl["nTx"] = pd.to_numeric(
            parts[1].str.replace("nTx=", "", regex=False),
            errors="coerce",
        )
    else:
        df_expl["nTx"] = pd.NA

    if parts.shape[1] > 2:
        df_expl["transcript_blob"] = parts[2]
    else:
        df_expl["transcript_blob"] = pd.NA

    tx_aware = df_expl["transcript_blob"].notna()

    # transcript-aware rows: keep only if row contains the top transcript
    df_tx = (
        df_expl[tx_aware]
        .assign(
            transcript_id=df_expl.loc[tx_aware, "transcript_blob"]
            .astype(str)
            .str.split(",")
        )
        .explode("transcript_id")
        .copy()
    )
    df_tx = df_tx[df_tx["transcript_id"].isin(top_tx_set)].copy()

    # gene-only rows: keep if gene has a top transcript, attach that transcript
    df_gene_only = df_expl[~tx_aware].copy()
    df_gene_only = df_gene_only[df_gene_only["gene_symbol"].isin(top_gene_set)].copy()

    df_gene_only = df_gene_only.merge(
        top_tx[["gene_symbol", "transcript_id"]],
        on="gene_symbol",
        how="left",
    )

    df_filt = pd.concat([df_tx, df_gene_only], ignore_index=True)

    # attach top transcript metadata/count
    df_filt = df_filt.merge(
        top_tx,
        on=["gene_symbol", "transcript_id"],
        how="left",
    )

    return df_filt


# ---------- run all samples ----------
all_merged = []

for sample in samples:
    print(f"\n=== {sample} ===")

    bed_path = (
        bedmethyl_base
        / sample
        / f"{sample}.reads.pass.modkit.coordA.cpg.chrX_annotated_with_CGI_shores_TSS.bed"
    )

    escapee_path = escapee_base / f"PBMC_sample-{sample}_escapee_overall.csv"

    if not bed_path.exists():
        print(f"SKIP missing bed: {bed_path}")
        continue
    if not escapee_path.exists():
        print(f"SKIP missing escapee: {escapee_path}")
        continue

    top_tx = get_top_tx_per_gene(sample)

    top_tx_out = outdir / f"{sample}_most_expressed_transcript_per_gene.tsv"
    top_tx.to_csv(top_tx_out, sep="\t", index=False)

    df = pd.read_csv(
        bed_path,
        sep="\t",
        header=None,
        names=bed_cols,
        dtype=bed_dtype,
    )

    df_m = df[df["modified base code"].str.lower().eq("m")].copy()

    df_filt = split_and_filter_bed_to_top_tx(df_m, top_tx)

    escapee = pd.read_csv(escapee_path)

    merged = df_filt.merge(
        escapee,
        how="left",
        left_on="gene_symbol",
        right_on="gene",
        suffixes=("", "_escapee"),
    )

    merged["sample"] = sample

    merged_gene_assigned = merged[
        merged["gene_symbol"].notna()
        & merged["gene_symbol"].astype(str).str.strip().ne(".")
    ].copy()

    out_path = outdir / f"{sample}_methylation_top_tx_escapee_merged.tsv.gz"
    merged_gene_assigned.to_csv(out_path, sep="\t", index=False, compression="gzip")

    print("m rows:", len(df_m))
    print("after top transcript filter:", len(df_filt))
    print("after escapee merge/gene assigned:", len(merged_gene_assigned))
    print("written:", out_path)

    all_merged.append(merged_gene_assigned)

# optional combined dataframe
if all_merged:
    combined = pd.concat(all_merged, ignore_index=True)
    combined_out = outdir / "all_samples_methylation_top_tx_escapee_merged.tsv.gz"
    combined.to_csv(combined_out, sep="\t", index=False, compression="gzip")
    print("\ncombined written:", combined_out)
    print("combined rows:", len(combined))
else:
    combined = pd.DataFrame()
    print("No samples processed.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# use the combined file from the previous step
in_path = Path("methylation_top_transcript_merged/all_samples_methylation_top_tx_escapee_merged.tsv.gz")

df = pd.read_csv(in_path, sep="\t", compression="gzip")

feature_col = "CG_feature" if "CG_feature" in df.columns else "CPG_feature"

df_plot = df[
    df["modified base code"].astype(str).str.lower().eq("m")
    & pd.to_numeric(df["fraction modified"], errors="coerce").notna()
    & df["label_overall"].notna()
    & df[feature_col].notna()
].copy()

df_plot["fraction modified"] = df_plot["fraction modified"].astype(float)

# optional: keep the main three feature classes
feature_order = ["CGI", "CGI_shore", "non-CGI"]
df_plot = df_plot[df_plot[feature_col].isin(feature_order)].copy()

# choose label order
label_order = sorted(df_plot["label_overall"].dropna().unique())

# facet rows = CPG feature, columns = sample
samples_order = sorted(df_plot["sample"].dropna().unique())

fig, axes = plt.subplots(
    nrows=len(feature_order),
    ncols=len(samples_order),
    figsize=(4.2 * len(samples_order), 3.6 * len(feature_order)),
    sharey=True,
)

# handle edge cases where one row/col
if len(feature_order) == 1 and len(samples_order) == 1:
    axes = [[axes]]
elif len(feature_order) == 1:
    axes = [axes]
elif len(samples_order) == 1:
    axes = [[ax] for ax in axes]

for i, feat in enumerate(feature_order):
    for j, sample in enumerate(samples_order):
        ax = axes[i][j]

        sub = df_plot[
            (df_plot[feature_col] == feat)
            & (df_plot["sample"] == sample)
        ].copy()

        data = [
            sub.loc[sub["label_overall"] == lab, "fraction modified"].values
            for lab in label_order
        ]

        # avoid crash if no data
        if sum(len(x) for x in data) == 0:
            ax.set_axis_off()
            continue

        ax.boxplot(data, labels=label_order, showfliers=False)

        if i == 0:
            ax.set_title(sample)

        if j == 0:
            ax.set_ylabel(f"{feat}\nFraction modified")
        else:
            ax.set_ylabel("")

        ax.tick_params(axis="x", rotation=35)

fig.suptitle(
    "Fraction modified by escapee class\nTop expressed transcript per gene only",
    y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(feature_order),
    figsize=(2 * len(feature_order), 4),
    sharey=True,
)

if len(feature_order) == 1:
    axes = [axes]

for ax, feat in zip(axes, feature_order):
    sub = df_plot[df_plot[feature_col] == feat].copy()

    data = [
        sub.loc[sub["label_overall"] == lab, "fraction modified"].values
        for lab in label_order
    ]

    ax.boxplot(data, labels=label_order, showfliers=False)
    ax.set_title(feat)
    ax.set_xlabel("label_overall")
    ax.tick_params(axis="x", rotation=35)

axes[0].set_ylabel("Fraction modified")

fig.suptitle("Fraction modified by escapee class, faceted by CpG feature")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

def p_to_stars(p):
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def add_sig_bar(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, color="black")
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=9, color="black")

fig, axes = plt.subplots(
    nrows=1,
    ncols=len(feature_order),
    figsize=(2.8 * len(feature_order), 4.6),
    sharey=True,
)

if len(feature_order) == 1:
    axes = [axes]

for ax, feat in zip(axes, feature_order):
    sub = df_plot[df_plot[feature_col] == feat].copy()

    data = [
        sub.loc[sub["label_overall"] == lab, "fraction modified"].dropna().astype(float).values
        for lab in label_order
    ]

    bp = ax.boxplot(
        data,
        labels=label_order,
        showfliers=False,
        patch_artist=False,
        boxprops=dict(color="black"),
        whiskerprops=dict(color="black"),
        capprops=dict(color="black"),
        medianprops=dict(color="black"),
    )

    ax.set_title(feat)
    ax.set_xlabel("label_overall")
    ax.tick_params(axis="x", rotation=35)

    nonempty = [d for d in data if len(d) > 0]
    if not nonempty:
        ax.set_axis_off()
        continue

    y_max = max(np.nanmax(d) for d in nonempty)
    y_min = min(np.nanmin(d) for d in nonempty)

    step = (y_max - y_min) * 0.10 if y_max > y_min else 0.05
    h = step * 0.35
    y = y_max + step

    # all pairwise comparisons
    for i, j in combinations(range(len(label_order)), 2):
        vals1 = data[i]
        vals2 = data[j]

        if len(vals1) == 0 or len(vals2) == 0:
            continue

        stat, p = mannwhitneyu(vals1, vals2, alternative="two-sided")
        stars = p_to_stars(p)

        add_sig_bar(ax, i + 1, j + 1, y, h, stars)
        y += step

    ax.set_ylim(top=y + step)

axes[0].set_ylabel("Fraction modified")

fig.suptitle("Fraction modified by escapee class, faceted by CpG feature")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

def p_to_stars(p):
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def add_sig_bar(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, color="black")
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=9, color="black")

feat = "CGI"

sub = df_plot[df_plot[feature_col] == feat].copy()

data = [
    sub.loc[sub["label_overall"] == lab, "fraction modified"]
       .dropna()
       .astype(float)
       .values
    for lab in label_order
]
label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}
fig, ax = plt.subplots(figsize=(3.2, 4.6))

display_labels = [label_map.get(x, x) for x in label_order]

ax.boxplot(
    data,
    labels=display_labels,
    showfliers=False,
    patch_artist=False,
    boxprops=dict(color="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
)

ax.set_title("CGI")
ax.set_xlabel("scDaisychain Gene Classification")
ax.set_ylabel("Percentage methylated")
ax.tick_params(axis="x", rotation=35)

nonempty = [d for d in data if len(d) > 0]

if nonempty:
    y_max = max(np.nanmax(d) for d in nonempty)
    y_min = min(np.nanmin(d) for d in nonempty)

    step = (y_max - y_min) * 0.10 if y_max > y_min else 0.05
    h = step * 0.35
    y = y_max + step

    for i, j in combinations(range(len(label_order)), 2):
        vals1 = data[i]
        vals2 = data[j]

        if len(vals1) == 0 or len(vals2) == 0:
            continue

        stat, p = mannwhitneyu(vals1, vals2, alternative="two-sided")
        stars = p_to_stars(p)

        add_sig_bar(ax, i + 1, j + 1, y, h, stars)
        y += step

    ax.set_ylim(top=y + step)

plt.title("Expressed CGI Promoters")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
from itertools import combinations
import pandas as pd
import numpy as np

df_gene_sample = (
    df_plot
    .dropna(subset=["sample", "gene_symbol", "label_overall", feature_col, "fraction modified"])
    .groupby(["sample", "gene_symbol", "label_overall", feature_col], as_index=False)
    .agg(
        methylation=("fraction modified", "median"),
        n_cpgs=("fraction modified", "size"),
    )
)

df_gene_sample = df_gene_sample[df_gene_sample["n_cpgs"] >= 3].copy()

In [ ]:
p_rows = []

for feat in feature_order:
    sub = df_gene_sample[df_gene_sample[feature_col] == feat].copy()

    groups = {
        lab: sub.loc[sub["label_overall"] == lab, "methylation"].dropna().astype(float).values
        for lab in label_order
    }

    for lab1, lab2 in combinations(label_order, 2):
        vals1 = groups[lab1]
        vals2 = groups[lab2]

        if len(vals1) == 0 or len(vals2) == 0:
            stat, p = pd.NA, pd.NA
        else:
            stat, p = mannwhitneyu(vals1, vals2, alternative="two-sided")

        p_rows.append({
            "CPG_feature": feat,
            "comparison": f"{lab1} vs {lab2}",
            "group1": lab1,
            "group2": lab2,
            "n_group1_gene_sample": len(vals1),
            "n_group2_gene_sample": len(vals2),
            "u_statistic": stat,
            "p_value": p,
            "stars": p_to_stars(p) if pd.notna(p) else pd.NA,
        })

pvals_gene_sample_df = pd.DataFrame(p_rows)
pvals_gene_sample_df

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd

model_rows = []

for feat in feature_order:
    sub = df_gene_sample[df_gene_sample[feature_col] == feat].copy()
    sub = sub.dropna(subset=["methylation", "label_overall", "sample"])

    sub["label_overall"] = pd.Categorical(
        sub["label_overall"],
        categories=["subject_to_XCI", "escapee", "uncertain"],
        ordered=False,
    )

    model = smf.mixedlm(
        "methylation ~ C(label_overall, Treatment(reference='subject_to_XCI'))",
        data=sub,
        groups=sub["sample"],
    )

    res = model.fit(method="lbfgs", reml=False)

    print("\n===", feat, "===")
    print(res.summary())

    for term, coef, pval in zip(res.params.index, res.params, res.pvalues):
        model_rows.append({
            "CPG_feature": feat,
            "term": term,
            "coef": coef,
            "p_value": pval,
        })

mixed_model_results = pd.DataFrame(model_rows)
mixed_model_results

In [ ]:
df_gene_sample = (
    df_plot
    .dropna(subset=["sample", "gene_symbol", "label_overall", feature_col, "fraction modified"])
    .groupby(["sample", "gene_symbol", "label_overall", feature_col], as_index=False)
    .agg(
        methylation=("fraction modified", "median"),
        n_cpgs=("fraction modified", "size"),
    )
)

df_gene_sample = df_gene_sample[df_gene_sample["n_cpgs"] >= 3].copy()
df_gene_sample.head()

In [ ]:
model_rows = []

for feat in feature_order:
    sub = df_gene_sample[df_gene_sample[feature_col] == feat].copy()
    sub = sub.dropna(subset=["methylation", "label_overall", "sample"])

    sub["label_overall"] = pd.Categorical(
        sub["label_overall"],
        categories=["subject_to_XCI", "escapee", "uncertain"],
        ordered=False,
    )

    model = smf.mixedlm(
        "methylation ~ C(label_overall, Treatment(reference='subject_to_XCI'))",
        data=sub,
        groups=sub["sample"],
    )

    res = model.fit(method="lbfgs", reml=False)

    print("\n===", feat, "===")
    print(res.summary())

    for term in res.params.index:
        model_rows.append({
            "CPG_feature": feat,
            "term": term,
            "coef": res.params.get(term, np.nan),
            "p_value": res.pvalues.get(term, np.nan),
        })

mixed_model_results = pd.DataFrame(model_rows)
mixed_model_results

In [ ]:
feat = "CGI"

sub = df_gene_sample[df_gene_sample[feature_col] == feat].copy()

data = [
    sub.loc[sub["label_overall"] == lab, "methylation"].dropna().astype(float).values
    for lab in label_order
]

fig, ax = plt.subplots(figsize=(3.4, 4.6))

ax.boxplot(
    data,
    labels=display_labels,
    showfliers=False,
    patch_artist=False,
    boxprops=dict(color="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
)

ax.set_title("CGI")
ax.set_xlabel("")
ax.set_ylabel("Percent modified")
ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Methylation by scDaisychain classification
#
# Makes plots for:
#   - CGI
#   - CGI shore
#   - non-CGI
#
# For each feature, makes:
#   1. all samples
#   2. controls only, excluding samples starting with "RG"
#
# Compact 50 mm × 60 mm version
# Legend on right-hand side
# No plot titles
#
# Font remains editable in Illustrator:
#   pdf.fonttype = 42
#   ps.fonttype = 42
#   svg.fonttype = "none"
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
from pathlib import Path

outdir = Path("plots")
outdir.mkdir(exist_ok=True)

# ---------- figure size ----------
MM_PER_INCH = 25.4
FIG_WIDTH_MM = 50
FIG_HEIGHT_MM = 60
FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "axes.titlesize": 6.2,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,
    "legend.fontsize": 4.6,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

label_order = ["escapee", "subject_to_XCI", "uncertain"]

label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}

display_labels = [label_map[x] for x in label_order]

feature_list = ["CGI", "CGI_shore", "non-CGI"]

pretty_feature_map = {
    "CGI": "CGI",
    "CGI_shore": "CGI shore",
    "non-CGI": "non-CGI",
}


def p_to_stars(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def add_sig_bar(ax, x1, x2, y, h, text):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=0.65,
        color="black",
        zorder=6,
        clip_on=False,
    )

    ax.text(
        (x1 + x2) / 2,
        y + h + 0.8,
        text,
        ha="center",
        va="bottom",
        fontsize=5.2,
        zorder=6,
        clip_on=False,
    )


def prepare_subset(df_gene_sample, feat, controls_only=False):
    sub = df_gene_sample[
        (df_gene_sample[feature_col] == feat)
        & df_gene_sample["label_overall"].isin(label_order)
    ].copy()

    sub = sub.dropna(
        subset=["methylation", "label_overall", "sample", "gene_symbol"]
    ).copy()

    sub["methylation"] = pd.to_numeric(sub["methylation"], errors="coerce")
    sub = sub.dropna(subset=["methylation"]).copy()

    if controls_only:
        sub = sub[
            ~sub["sample"].astype(str).str.startswith("RG")
        ].copy()

    sub["label_overall"] = pd.Categorical(
        sub["label_overall"],
        categories=["subject_to_XCI", "escapee", "uncertain"],
    )

    return sub


def run_rlm_pairwise(sub, feat):
    rlm = smf.rlm(
        "methylation ~ C(label_overall, Treatment(reference='subject_to_XCI')) + C(sample)",
        data=sub,
    ).fit()

    rlm_terms = pd.DataFrame({
        "feature": feat,
        "term": rlm.params.index,
        "coef": rlm.params.values,
        "std_err": rlm.bse.values,
        "z_value": rlm.tvalues.values,
        "p_value": rlm.pvalues.values,
        "n_gene_sample_rows": int(rlm.nobs),
        "n_samples": sub["sample"].nunique(),
        "n_genes": sub["gene_symbol"].nunique(),
    })

    params = rlm.params
    cov = rlm.cov_params()

    term_escapee = "C(label_overall, Treatment(reference='subject_to_XCI'))[T.escapee]"
    term_uncertain = "C(label_overall, Treatment(reference='subject_to_XCI'))[T.uncertain]"

    contrasts = [
        (
            "Escapee vs Subject to XCI",
            "escapee",
            "subject_to_XCI",
            {term_escapee: 1},
        ),
        (
            "Unclassifiable vs Subject to XCI",
            "uncertain",
            "subject_to_XCI",
            {term_uncertain: 1},
        ),
        (
            "Escapee vs Unclassifiable",
            "escapee",
            "uncertain",
            {term_escapee: 1, term_uncertain: -1},
        ),
    ]

    rows = []

    for comparison, group1, group2, contrast in contrasts:
        L = pd.Series(0.0, index=params.index)

        for term, weight in contrast.items():
            L[term] = weight

        coef = float(np.dot(L, params))
        se = float(np.sqrt(np.dot(L, np.dot(cov, L))))
        z = coef / se
        p = 2 * norm.sf(abs(z))

        rows.append({
            "feature": feat,
            "method": "robust_linear_model_sample_fixed_effects",
            "comparison": comparison,
            "group1": group1,
            "group2": group2,
            "coef_group1_minus_group2": coef,
            "std_err": se,
            "z_value": z,
            "p_value": p,
            "n_gene_sample_rows": int(rlm.nobs),
            "n_samples": sub["sample"].nunique(),
            "n_genes": sub["gene_symbol"].nunique(),
        })

    rlm_pairwise = pd.DataFrame(rows)

    rlm_pairwise["p_adj_fdr"] = multipletests(
        rlm_pairwise["p_value"],
        method="fdr_bh",
    )[1]

    rlm_pairwise["p_adj_bonferroni"] = multipletests(
        rlm_pairwise["p_value"],
        method="bonferroni",
    )[1]

    rlm_pairwise["stars_fdr"] = rlm_pairwise["p_adj_fdr"].apply(p_to_stars)

    return rlm_terms, rlm_pairwise


def plot_methylation_by_class(
    sub,
    rlm_pairwise,
    feat,
    stem,
):
    data = [
        sub.loc[sub["label_overall"] == lab, "methylation"]
           .dropna()
           .astype(float)
           .values
        for lab in label_order
    ]

    samples_order = sorted(sub["sample"].dropna().unique())

    sample_colors = dict(
        zip(
            samples_order,
            plt.cm.tab10(np.linspace(0, 1, len(samples_order))),
        )
    )

    fig, ax = plt.subplots(figsize=FIGSIZE)

    rng = np.random.default_rng(1)

    # ---------- points behind ----------
    for x, lab in enumerate(label_order, start=1):
        sub_lab = sub[sub["label_overall"] == lab].copy()

        for sample in samples_order:
            vals = (
                sub_lab.loc[sub_lab["sample"] == sample, "methylation"]
                .dropna()
                .astype(float)
                .values
            )

            if len(vals) == 0:
                continue

            jitter = rng.uniform(-0.20, 0.20, size=len(vals))

            ax.scatter(
                np.full(len(vals), x) + jitter,
                vals,
                s=4.4,
                color=sample_colors[sample],
                alpha=0.35,
                linewidths=0,
                zorder=1,
                rasterized=True,
            )

    # ---------- violin outlines only ----------
    vp = ax.violinplot(
        data,
        positions=np.arange(1, len(label_order) + 1),
        widths=0.76,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    for body in vp["bodies"]:
        body.set_facecolor("none")
        body.set_edgecolor("black")
        body.set_alpha(1)
        body.set_linewidth(0.75)
        body.set_zorder(3)

    # ---------- inner boxplot outlines only ----------
    ax.boxplot(
        data,
        positions=np.arange(1, len(label_order) + 1),
        widths=0.20,
        showfliers=False,
        patch_artist=True,
        boxprops=dict(facecolor="none", color="black", linewidth=0.75),
        whiskerprops=dict(color="black", linewidth=0.75),
        capprops=dict(color="black", linewidth=0.75),
        medianprops=dict(color="black", linewidth=0.9),
        zorder=4,
    )

    # ---------- significance bars ----------
    nonempty = [d for d in data if len(d) > 0]

    plot_ymax = 140

    if len(nonempty) > 0:
        y_max_data = max(np.nanmax(d) for d in nonempty)
    else:
        y_max_data = 100

    bar_start = min(y_max_data + 5, 108)
    bar_step = 9
    bar_h = 3
    y = bar_start

    for _, row in rlm_pairwise.iterrows():
        if pd.isna(row["p_adj_fdr"]):
            continue

        x1 = label_order.index(row["group1"]) + 1
        x2 = label_order.index(row["group2"]) + 1

        add_sig_bar(ax, x1, x2, y, bar_h, row["stars_fdr"])
        y += bar_step

    # ---------- axes ----------
    ax.set_xlim(0.50, len(label_order) + 0.50)
    ax.set_ylim(0, plot_ymax)

    ax.set_xticks(np.arange(1, len(label_order) + 1))
    ax.set_xticklabels(
        display_labels,
        rotation=35,
        ha="right",
        rotation_mode="anchor",
    )

    ax.set_yticks(np.arange(0, 101, 20))

    ax.set_ylabel("Percent methylated", labelpad=1.8)

    # No plot title
    # Feature is encoded in output filename.
    # Add a tiny in-panel label only if you want:
    # ax.text(0.02, 0.98, pretty_feature_map.get(feat, feat),
    #         transform=ax.transAxes, ha="left", va="top",
    #         fontsize=5.5, fontweight="bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(axis="y", alpha=0.22, linewidth=0.4)
    ax.set_axisbelow(True)

    ax.tick_params(axis="x", pad=1.5)
    ax.tick_params(axis="y", pad=1.2)

    # ---------- legend on right-hand side ----------
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=sample_colors[s],
            markeredgewidth=0,
            label=s,
            markersize=3.1,
            alpha=0.8,
        )
        for s in samples_order
    ]

    ax.legend(
        handles=handles,
        title="Sample",
        title_fontsize=4.7,
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.50),
        borderaxespad=0,
        handlelength=0.8,
        handletextpad=0.35,
        labelspacing=0.15,
        fontsize=4.5,
        ncol=1,
    )

    # ---------- transparent background ----------
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    # Leave space on right for legend and bottom for rotated labels
    fig.subplots_adjust(
        left=0.24,
        right=0.72,
        bottom=0.22,
        top=0.98,
    )

    out_base = outdir / stem

    for ext in ["png", "pdf", "svg", "eps"]:
        fig.savefig(
            f"{out_base}.{ext}",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.03,
            transparent=True,
        )

    plt.show()


def make_feature_plots(df_gene_sample, controls_only=False):
    suffix = "controls_only" if controls_only else "all_samples"

    all_rlm_pairwise = []
    all_rlm_terms = []

    for feat in feature_list:
        print(f"\n=== {feat} | {suffix} ===")

        sub = prepare_subset(
            df_gene_sample=df_gene_sample,
            feat=feat,
            controls_only=controls_only,
        )

        if sub.empty:
            print(f"Skipping {feat} {suffix}: no data after filtering.")
            continue

        rlm_terms, rlm_pairwise = run_rlm_pairwise(sub, feat)

        rlm_terms["sample_set"] = suffix
        rlm_pairwise["sample_set"] = suffix

        all_rlm_terms.append(rlm_terms)
        all_rlm_pairwise.append(rlm_pairwise)

        display(rlm_pairwise)

        safe_feat = feat.replace("-", "_").replace("/", "_")

        stem = (
            f"{safe_feat}_methylation_by_XCI_class_gene_sample_"
            f"RLM_FDR_{suffix}_50x60mm"
        )

        plot_methylation_by_class(
            sub=sub,
            rlm_pairwise=rlm_pairwise,
            feat=feat,
            stem=stem,
        )

    return all_rlm_terms, all_rlm_pairwise


# ============================================================
# Make all-sample and controls-only plots
# ============================================================

all_terms_all, all_pairwise_all = make_feature_plots(
    df_gene_sample=df_gene_sample,
    controls_only=False,
)

all_terms_controls, all_pairwise_controls = make_feature_plots(
    df_gene_sample=df_gene_sample,
    controls_only=True,
)


# ============================================================
# Save combined model results
# ============================================================

rlm_pairwise_all_features = pd.concat(
    all_pairwise_all + all_pairwise_controls,
    ignore_index=True,
)

rlm_terms_all_features = pd.concat(
    all_terms_all + all_terms_controls,
    ignore_index=True,
)

rlm_pairwise_all_features.to_csv(
    outdir / "all_CPG_features_methylation_RLM_pairwise_all_and_controls.tsv",
    sep="\t",
    index=False,
)

rlm_terms_all_features.to_csv(
    outdir / "all_CPG_features_methylation_RLM_all_terms_all_and_controls.tsv",
    sep="\t",
    index=False,
)

display(rlm_pairwise_all_features)

In [ ]:
# ============================================================
# Methylation by scDaisychain classification
#
# Makes plots for:
#   - CGI
#   - CGI shore
#   - non-CGI
#
# For each feature, makes:
#   1. all samples
#   2. controls only, excluding samples starting with "RG"
#
# Compact 50 mm × 60 mm version
# No plot titles
# Boxplot + monochrome points
#
# Font remains editable in Illustrator:
#   pdf.fonttype = 42
#   ps.fonttype = 42
#   svg.fonttype = "none"
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
from pathlib import Path

outdir = Path("plots")
outdir.mkdir(exist_ok=True)

# ---------- figure size ----------
MM_PER_INCH = 25.4
FIG_WIDTH_MM = 50
FIG_HEIGHT_MM = 60
FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "axes.titlesize": 6.2,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,
    "legend.fontsize": 4.6,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

label_order = ["escapee", "subject_to_XCI", "uncertain"]

label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}

display_labels = [label_map[x] for x in label_order]

feature_list = ["CGI", "CGI_shore", "non-CGI"]

pretty_feature_map = {
    "CGI": "CGI",
    "CGI_shore": "CGI shore",
    "non-CGI": "non-CGI",
}


def p_to_stars(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def add_sig_bar(ax, x1, x2, y, h, text):
    x1, x2 = sorted([x1, x2])

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=0.65,
        color="black",
        zorder=6,
        clip_on=False,
    )

    ax.text(
        (x1 + x2) / 2,
        y + h + 0.8,
        text,
        ha="center",
        va="bottom",
        fontsize=5.2,
        zorder=6,
        clip_on=False,
    )


def prepare_subset(df_gene_sample, feat, controls_only=False):
    sub = df_gene_sample[
        (df_gene_sample[feature_col] == feat)
        & df_gene_sample["label_overall"].isin(label_order)
    ].copy()

    sub = sub.dropna(
        subset=["methylation", "label_overall", "sample", "gene_symbol"]
    ).copy()

    sub["methylation"] = pd.to_numeric(sub["methylation"], errors="coerce")
    sub = sub.dropna(subset=["methylation"]).copy()

    if controls_only:
        sub = sub[
            ~sub["sample"].astype(str).str.startswith("RG")
        ].copy()

    sub["label_overall"] = pd.Categorical(
        sub["label_overall"],
        categories=["subject_to_XCI", "escapee", "uncertain"],
    )

    return sub


def run_rlm_pairwise(sub, feat):
    rlm = smf.rlm(
        "methylation ~ C(label_overall, Treatment(reference='subject_to_XCI')) + C(sample)",
        data=sub,
    ).fit()

    rlm_terms = pd.DataFrame({
        "feature": feat,
        "term": rlm.params.index,
        "coef": rlm.params.values,
        "std_err": rlm.bse.values,
        "z_value": rlm.tvalues.values,
        "p_value": rlm.pvalues.values,
        "n_gene_sample_rows": int(rlm.nobs),
        "n_samples": sub["sample"].nunique(),
        "n_genes": sub["gene_symbol"].nunique(),
    })

    params = rlm.params
    cov = rlm.cov_params()

    term_escapee = "C(label_overall, Treatment(reference='subject_to_XCI'))[T.escapee]"
    term_uncertain = "C(label_overall, Treatment(reference='subject_to_XCI'))[T.uncertain]"

    contrasts = [
        (
            "Escapee vs Subject to XCI",
            "escapee",
            "subject_to_XCI",
            {term_escapee: 1},
        ),
        (
            "Unclassifiable vs Subject to XCI",
            "uncertain",
            "subject_to_XCI",
            {term_uncertain: 1},
        ),
        (
            "Escapee vs Unclassifiable",
            "escapee",
            "uncertain",
            {term_escapee: 1, term_uncertain: -1},
        ),
    ]

    rows = []

    for comparison, group1, group2, contrast in contrasts:
        L = pd.Series(0.0, index=params.index)

        for term, weight in contrast.items():
            L[term] = weight

        coef = float(np.dot(L, params))
        se = float(np.sqrt(np.dot(L, np.dot(cov, L))))
        z = coef / se
        p = 2 * norm.sf(abs(z))

        rows.append({
            "feature": feat,
            "method": "robust_linear_model_sample_fixed_effects",
            "comparison": comparison,
            "group1": group1,
            "group2": group2,
            "coef_group1_minus_group2": coef,
            "std_err": se,
            "z_value": z,
            "p_value": p,
            "n_gene_sample_rows": int(rlm.nobs),
            "n_samples": sub["sample"].nunique(),
            "n_genes": sub["gene_symbol"].nunique(),
        })

    rlm_pairwise = pd.DataFrame(rows)

    rlm_pairwise["p_adj_fdr"] = multipletests(
        rlm_pairwise["p_value"],
        method="fdr_bh",
    )[1]

    rlm_pairwise["p_adj_bonferroni"] = multipletests(
        rlm_pairwise["p_value"],
        method="bonferroni",
    )[1]

    rlm_pairwise["stars_fdr"] = rlm_pairwise["p_adj_fdr"].apply(p_to_stars)

    return rlm_terms, rlm_pairwise


def plot_methylation_by_class(
    sub,
    rlm_pairwise,
    feat,
    stem,
):
    data = [
        sub.loc[sub["label_overall"] == lab, "methylation"]
           .dropna()
           .astype(float)
           .values
        for lab in label_order
    ]

    fig, ax = plt.subplots(figsize=FIGSIZE)

    positions = np.arange(1, len(label_order) + 1)
    rng = np.random.default_rng(1)

    # ---------- monochrome points behind ----------
    for x, lab in enumerate(label_order, start=1):
        vals = (
            sub.loc[sub["label_overall"] == lab, "methylation"]
            .dropna()
            .astype(float)
            .values
        )

        if len(vals) == 0:
            continue

        jitter = rng.uniform(-0.18, 0.18, size=len(vals))

        ax.scatter(
            np.full(len(vals), x) + jitter,
            vals,
            s=4.5,
            color="0.35",
            alpha=0.45,
            linewidths=0,
            zorder=1,
            rasterized=True,
        )

    # ---------- boxplot on top ----------
    ax.boxplot(
        data,
        positions=positions,
        widths=0.52,
        showfliers=False,
        patch_artist=True,
        boxprops=dict(
            facecolor="none",
            edgecolor="black",
            linewidth=0.8,
        ),
        whiskerprops=dict(
            color="black",
            linewidth=0.8,
        ),
        capprops=dict(
            color="black",
            linewidth=0.8,
        ),
        medianprops=dict(
            color="black",
            linewidth=1.0,
        ),
        zorder=4,
    )

    # ---------- significance bars ----------
    nonempty = [d for d in data if len(d) > 0]

    plot_ymax = 140

    if len(nonempty) > 0:
        y_max_data = max(np.nanmax(d) for d in nonempty)
    else:
        y_max_data = 100

    bar_start = min(y_max_data + 5, 108)
    bar_step = 9
    bar_h = 3
    y = bar_start

    for _, row in rlm_pairwise.iterrows():
        if pd.isna(row["p_adj_fdr"]):
            continue

        x1 = label_order.index(row["group1"]) + 1
        x2 = label_order.index(row["group2"]) + 1

        add_sig_bar(ax, x1, x2, y, bar_h, row["stars_fdr"])
        y += bar_step

    # ---------- axes ----------
    ax.set_xlim(0.50, len(label_order) + 0.50)
    ax.set_ylim(0, plot_ymax)

    ax.set_xticks(positions)
    ax.set_xticklabels(
        display_labels,
        rotation=35,
        ha="right",
        rotation_mode="anchor",
    )

    ax.set_yticks(np.arange(0, 101, 20))

    ax.set_ylabel("Percent methylated", labelpad=1.8)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(axis="y", alpha=0.22, linewidth=0.4)
    ax.set_axisbelow(True)

    ax.tick_params(axis="x", pad=1.5)
    ax.tick_params(axis="y", pad=1.2)

    # ---------- transparent background ----------
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    fig.subplots_adjust(
        left=0.24,
        right=0.98,
        bottom=0.22,
        top=0.98,
    )

    out_base = outdir / stem

    for ext in ["png", "pdf", "svg", "eps"]:
        fig.savefig(
            f"{out_base}.{ext}",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.03,
            transparent=True,
        )

    plt.show()


def make_feature_plots(df_gene_sample, controls_only=False):
    suffix = "controls_only" if controls_only else "all_samples"

    all_rlm_pairwise = []
    all_rlm_terms = []

    for feat in feature_list:
        print(f"\n=== {feat} | {suffix} ===")

        sub = prepare_subset(
            df_gene_sample=df_gene_sample,
            feat=feat,
            controls_only=controls_only,
        )

        if sub.empty:
            print(f"Skipping {feat} {suffix}: no data after filtering.")
            continue

        rlm_terms, rlm_pairwise = run_rlm_pairwise(sub, feat)

        rlm_terms["sample_set"] = suffix
        rlm_pairwise["sample_set"] = suffix

        all_rlm_terms.append(rlm_terms)
        all_rlm_pairwise.append(rlm_pairwise)

        display(rlm_pairwise)

        safe_feat = feat.replace("-", "_").replace("/", "_")

        stem = (
            f"{safe_feat}_methylation_by_XCI_class_gene_sample_"
            f"boxplot_points_monochrome_RLM_FDR_{suffix}_50x60mm"
        )

        plot_methylation_by_class(
            sub=sub,
            rlm_pairwise=rlm_pairwise,
            feat=feat,
            stem=stem,
        )

    return all_rlm_terms, all_rlm_pairwise


# ============================================================
# Make all-sample and controls-only plots
# ============================================================

all_terms_all, all_pairwise_all = make_feature_plots(
    df_gene_sample=df_gene_sample,
    controls_only=False,
)

all_terms_controls, all_pairwise_controls = make_feature_plots(
    df_gene_sample=df_gene_sample,
    controls_only=True,
)


# ============================================================
# Save combined model results
# ============================================================

rlm_pairwise_all_features = pd.concat(
    all_pairwise_all + all_pairwise_controls,
    ignore_index=True,
)

rlm_terms_all_features = pd.concat(
    all_terms_all + all_terms_controls,
    ignore_index=True,
)

rlm_pairwise_all_features.to_csv(
    outdir / "all_CPG_features_methylation_RLM_pairwise_all_and_controls.tsv",
    sep="\t",
    index=False,
)

rlm_terms_all_features.to_csv(
    outdir / "all_CPG_features_methylation_RLM_all_terms_all_and_controls.tsv",
    sep="\t",
    index=False,
)

display(rlm_pairwise_all_features)

In [ ]:
# ============================================================
# Methylation by scDaisychain classification
#
# Makes plots for:
#   - CGI
#   - CGI shore
#   - non-CGI
#
# For each feature, makes:
#   1. all samples
#   2. controls only, excluding samples starting with "RG"
#
# For each sample set, makes FOUR versions:
#   A. no unclassifiable  + points
#   B. no unclassifiable  + no points
#   C. with unclassifiable + points
#   D. with unclassifiable + no points
#
# Compact 50 mm × 60 mm version
# No plot titles
#
# Font remains editable in Illustrator:
#   pdf.fonttype = 42
#   ps.fonttype = 42
#   svg.fonttype = "none"
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
from pathlib import Path

outdir = Path("plots")
outdir.mkdir(exist_ok=True)

# ---------- figure size ----------
MM_PER_INCH = 25.4
FIG_WIDTH_MM = 50
FIG_HEIGHT_MM = 60
FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "axes.titlesize": 6.2,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,
    "legend.fontsize": 4.6,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

feature_list = ["CGI", "CGI_shore", "non-CGI"]

pretty_feature_map = {
    "CGI": "CGI",
    "CGI_shore": "CGI shore",
    "non-CGI": "non-CGI",
}

label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}


def get_label_order(include_unclassified):
    if include_unclassified:
        return ["escapee", "subject_to_XCI", "uncertain"]
    else:
        return ["escapee", "subject_to_XCI"]


def p_to_stars(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def add_sig_bar(ax, x1, x2, y, h, text):
    x1, x2 = sorted([x1, x2])

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=0.65,
        color="black",
        zorder=6,
        clip_on=False,
    )

    ax.text(
        (x1 + x2) / 2,
        y + h + 0.8,
        text,
        ha="center",
        va="bottom",
        fontsize=5.2,
        zorder=6,
        clip_on=False,
    )


def prepare_subset(
    df_gene_sample,
    feat,
    controls_only=False,
    include_unclassified=False,
):
    label_order = get_label_order(include_unclassified)

    sub = df_gene_sample[
        (df_gene_sample[feature_col] == feat)
        & df_gene_sample["label_overall"].isin(label_order)
    ].copy()

    sub = sub.dropna(
        subset=["methylation", "label_overall", "sample", "gene_symbol"]
    ).copy()

    sub["methylation"] = pd.to_numeric(sub["methylation"], errors="coerce")
    sub = sub.dropna(subset=["methylation"]).copy()

    if controls_only:
        sub = sub[
            ~sub["sample"].astype(str).str.startswith("RG")
        ].copy()

    # Keep subject_to_XCI as the regression reference
    reg_categories = ["subject_to_XCI", "escapee"]
    if include_unclassified:
        reg_categories.append("uncertain")

    sub["label_overall"] = pd.Categorical(
        sub["label_overall"],
        categories=reg_categories,
    )

    return sub


def run_rlm_pairwise(sub, feat, include_unclassified=False):
    formula = (
        "methylation ~ "
        "C(label_overall, Treatment(reference='subject_to_XCI')) + "
        "C(sample)"
    )

    rlm = smf.rlm(formula, data=sub).fit()

    rlm_terms = pd.DataFrame({
        "feature": feat,
        "term": rlm.params.index,
        "coef": rlm.params.values,
        "std_err": rlm.bse.values,
        "z_value": rlm.tvalues.values,
        "p_value": rlm.pvalues.values,
        "n_gene_sample_rows": int(rlm.nobs),
        "n_samples": sub["sample"].nunique(),
        "n_genes": sub["gene_symbol"].nunique(),
    })

    params = rlm.params
    cov = rlm.cov_params()

    term_escapee = (
        "C(label_overall, Treatment(reference='subject_to_XCI'))[T.escapee]"
    )
    term_uncertain = (
        "C(label_overall, Treatment(reference='subject_to_XCI'))[T.uncertain]"
    )

    if include_unclassified:
        contrasts = [
            (
                "Escapee vs Subject to XCI",
                "escapee",
                "subject_to_XCI",
                {term_escapee: 1},
            ),
            (
                "Unclassifiable vs Subject to XCI",
                "uncertain",
                "subject_to_XCI",
                {term_uncertain: 1},
            ),
            (
                "Escapee vs Unclassifiable",
                "escapee",
                "uncertain",
                {term_escapee: 1, term_uncertain: -1},
            ),
        ]
    else:
        contrasts = [
            (
                "Escapee vs Subject to XCI",
                "escapee",
                "subject_to_XCI",
                {term_escapee: 1},
            )
        ]

    rows = []

    for comparison, group1, group2, contrast in contrasts:
        L = pd.Series(0.0, index=params.index)

        for term, weight in contrast.items():
            if term in L.index:
                L[term] = weight

        coef = float(np.dot(L, params))
        se = float(np.sqrt(np.dot(L, np.dot(cov, L))))
        z = coef / se
        p = 2 * norm.sf(abs(z))

        rows.append({
            "feature": feat,
            "method": "robust_linear_model_sample_fixed_effects",
            "comparison": comparison,
            "group1": group1,
            "group2": group2,
            "coef_group1_minus_group2": coef,
            "std_err": se,
            "z_value": z,
            "p_value": p,
            "n_gene_sample_rows": int(rlm.nobs),
            "n_samples": sub["sample"].nunique(),
            "n_genes": sub["gene_symbol"].nunique(),
        })

    rlm_pairwise = pd.DataFrame(rows)

    if len(rlm_pairwise) > 0:
        rlm_pairwise["p_adj_fdr"] = multipletests(
            rlm_pairwise["p_value"],
            method="fdr_bh",
        )[1]

        rlm_pairwise["p_adj_bonferroni"] = multipletests(
            rlm_pairwise["p_value"],
            method="bonferroni",
        )[1]

        rlm_pairwise["stars_fdr"] = rlm_pairwise["p_adj_fdr"].apply(p_to_stars)

    return rlm_terms, rlm_pairwise


def plot_methylation_by_class(
    sub,
    rlm_pairwise,
    feat,
    stem,
    include_unclassified=False,
    show_points=True,
):
    label_order = get_label_order(include_unclassified)
    display_labels = [label_map[x] for x in label_order]

    data = [
        sub.loc[sub["label_overall"] == lab, "methylation"]
           .dropna()
           .astype(float)
           .values
        for lab in label_order
    ]

    fig, ax = plt.subplots(figsize=FIGSIZE)

    positions = np.arange(1, len(label_order) + 1)
    rng = np.random.default_rng(1)

    # ---------- monochrome points behind ----------
    if show_points:
        for x, lab in enumerate(label_order, start=1):
            vals = (
                sub.loc[sub["label_overall"] == lab, "methylation"]
                .dropna()
                .astype(float)
                .values
            )

            if len(vals) == 0:
                continue

            jitter = rng.uniform(-0.18, 0.18, size=len(vals))

            ax.scatter(
                np.full(len(vals), x) + jitter,
                vals,
                s=6.0,
                color="0.35",
                alpha=0.55,
                linewidths=0,
                zorder=1,
                rasterized=True,
                clip_on=False,
            )

    # ---------- boxplot ----------
    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.52,
        showfliers=False,
        patch_artist=True,
        boxprops=dict(
            facecolor="none",
            edgecolor="black",
            linewidth=0.8,
        ),
        whiskerprops=dict(
            color="black",
            linewidth=0.8,
        ),
        capprops=dict(
            color="black",
            linewidth=0.8,
        ),
        medianprops=dict(
            color="black",
            linewidth=1.0,
        ),
        zorder=4,
    )

    # ---------- dynamic significance bars + y-limit ----------
    bars_to_draw = rlm_pairwise[~rlm_pairwise["p_adj_fdr"].isna()].copy()
    n_bars = len(bars_to_draw)

    bar_h = 3
    bar_step = 8

    if show_points:
        nonempty = [d for d in data if len(d) > 0]

        if len(nonempty) > 0:
            top_visible = max(np.nanmax(d) for d in nonempty)
        else:
            top_visible = 100

        bar_start = top_visible + 5
        top_needed = (
            bar_start
            + max(0, n_bars - 1) * bar_step
            + bar_h
            + 8
        )

        plot_ymax = np.ceil(top_needed / 10) * 10
        plot_ymax = max(40, min(plot_ymax, 140))
        y_min = -3

    else:
        whisker_tops = [np.max(w.get_ydata()) for w in bp["whiskers"]]

        if len(whisker_tops) > 0:
            top_visible = max(whisker_tops)
        else:
            top_visible = 100

        bar_start = top_visible + 2
        top_needed = (
            bar_start
            + max(0, n_bars - 1) * bar_step
            + bar_h
            + 6
        )

        plot_ymax = np.ceil(top_needed / 10) * 10
        plot_ymax = max(30, min(plot_ymax, 120))
        y_min = 0

    y = bar_start
    for _, row in bars_to_draw.iterrows():
        x1 = label_order.index(row["group1"]) + 1
        x2 = label_order.index(row["group2"]) + 1

        add_sig_bar(
            ax,
            x1=x1,
            x2=x2,
            y=y,
            h=bar_h,
            text=row["stars_fdr"],
        )
        y += bar_step

    # ---------- axes ----------
    ax.set_xlim(0.50, len(label_order) + 0.50)
    ax.set_ylim(y_min, plot_ymax)

    ax.set_xticks(positions)
    ax.set_xticklabels(
        display_labels,
        rotation=30,
        ha="right",
        rotation_mode="anchor",
    )

    tick_ymax = min(plot_ymax, 100)

    ax.set_yticks(
        np.arange(0, tick_ymax + 1, 20)
    )
    feature_label = pretty_feature_map.get(feat, feat)

    ax.set_ylabel(
        f"Percent methylated per {feature_label} promoter",
        labelpad=1.8,
    )
    ax.set_xlabel("scDaisychain XCI classification")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(axis="y", alpha=0.22, linewidth=0.4)
    ax.set_axisbelow(True)

    ax.tick_params(axis="x", pad=1.5)
    ax.tick_params(axis="y", pad=1.2)

    # ---------- transparent background ----------
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    fig.subplots_adjust(
        left=0.24,
        right=0.98,
        bottom=0.20,
        top=0.98,
    )

    out_base = outdir / stem

    for ext in ["png", "pdf", "svg", "eps"]:
        fig.savefig(
            f"{out_base}.{ext}",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.03,
            transparent=True,
        )

    plt.show()


def make_feature_plots(
    df_gene_sample,
    controls_only=False,
    include_unclassified=False,
):
    suffix = "controls_only" if controls_only else "all_samples"
    class_suffix = (
        "with_unclassifiable"
        if include_unclassified
        else "no_unclassifiable"
    )

    label_order = get_label_order(include_unclassified)

    all_rlm_pairwise = []
    all_rlm_terms = []

    for feat in feature_list:
        print(f"\n=== {feat} | {suffix} | {class_suffix} ===")

        sub = prepare_subset(
            df_gene_sample=df_gene_sample,
            feat=feat,
            controls_only=controls_only,
            include_unclassified=include_unclassified,
        )

        if sub.empty:
            print(
                f"Skipping {feat} {suffix} {class_suffix}: "
                f"no data after filtering."
            )
            continue

        counts = sub["label_overall"].value_counts()

        missing_groups = [
            lab for lab in label_order
            if counts.get(lab, 0) == 0
        ]

        if len(missing_groups) > 0:
            print(
                f"Skipping {feat} {suffix} {class_suffix}: "
                f"missing groups {missing_groups} after filtering."
            )
            continue

        rlm_terms, rlm_pairwise = run_rlm_pairwise(
            sub=sub,
            feat=feat,
            include_unclassified=include_unclassified,
        )

        rlm_terms["sample_set"] = suffix
        rlm_terms["class_set"] = class_suffix

        rlm_pairwise["sample_set"] = suffix
        rlm_pairwise["class_set"] = class_suffix

        all_rlm_terms.append(rlm_terms)
        all_rlm_pairwise.append(rlm_pairwise)

        display(rlm_pairwise)

        safe_feat = feat.replace("-", "_").replace("/", "_")

        # ---------- version 1: with points ----------
        stem_points = (
            f"{safe_feat}_methylation_by_XCI_class_gene_sample_"
            f"boxplot_points_monochrome_{class_suffix}_"
            f"RLM_FDR_{suffix}_50x60mm"
        )

        plot_methylation_by_class(
            sub=sub,
            rlm_pairwise=rlm_pairwise,
            feat=feat,
            stem=stem_points,
            include_unclassified=include_unclassified,
            show_points=True,
        )

        # ---------- version 2: no points ----------
        stem_no_points = (
            f"{safe_feat}_methylation_by_XCI_class_gene_sample_"
            f"boxplot_no_points_{class_suffix}_"
            f"RLM_FDR_{suffix}_50x60mm"
        )

        plot_methylation_by_class(
            sub=sub,
            rlm_pairwise=rlm_pairwise,
            feat=feat,
            stem=stem_no_points,
            include_unclassified=include_unclassified,
            show_points=False,
        )

    return all_rlm_terms, all_rlm_pairwise


# ============================================================
# Make all plot sets
# ============================================================

all_results_terms = []
all_results_pairwise = []

for controls_only in [False, True]:
    for include_unclassified in [False, True]:
        terms_list, pairwise_list = make_feature_plots(
            df_gene_sample=df_gene_sample,
            controls_only=controls_only,
            include_unclassified=include_unclassified,
        )
        all_results_terms.extend(terms_list)
        all_results_pairwise.extend(pairwise_list)


# ============================================================
# Save combined model results
# ============================================================

if len(all_results_pairwise) > 0:
    rlm_pairwise_all_features = pd.concat(
        all_results_pairwise,
        ignore_index=True,
    )

    rlm_pairwise_all_features.to_csv(
        outdir / "all_CPG_features_methylation_RLM_pairwise_all_plotsets.tsv",
        sep="\t",
        index=False,
    )

    display(rlm_pairwise_all_features)
else:
    print("No pairwise RLM results to save.")

if len(all_results_terms) > 0:
    rlm_terms_all_features = pd.concat(
        all_results_terms,
        ignore_index=True,
    )

    rlm_terms_all_features.to_csv(
        outdir / "all_CPG_features_methylation_RLM_all_terms_all_plotsets.tsv",
        sep="\t",
        index=False,
    )

    display(rlm_terms_all_features)
else:
    print("No RLM term results to save.")

In [ ]:
# ============================================================
# Methylation by scDaisychain classification
#
# Dot plots only
# No Unclassifiable
# Faceted by donor: all donors in one row
#
# Makes plots for:
#   - CGI
#   - CGI shore
#   - non-CGI
#
# For each feature, makes:
#   1. all samples
#   2. controls only, excluding samples starting with "RG"
#
# Labels:
#   subject_to_XCI -> Inactive
#   escapee        -> escape
#
# Per-donor test:
#   methylation ~ label_overall
#   escape vs Inactive
#   FDR corrected across donors within each plot
#
# Font remains editable in Illustrator:
#   pdf.fonttype = 42
#   ps.fonttype = 42
#   svg.fonttype = "none"
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

outdir = Path("plots")
outdir.mkdir(exist_ok=True)

# ---------- figure/font settings ----------
MM_PER_INCH = 25.4

# One row of donors.
# All samples: 8 donors -> 144 x 38 mm
# Controls only: 4 donors -> 72 x 38 mm
PANEL_WIDTH_MM = 18
PANEL_HEIGHT_MM = 38

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "axes.titlesize": 6.2,
    "xtick.labelsize": 5.0,
    "ytick.labelsize": 5.2,
    "legend.fontsize": 4.8,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

feature_list = ["CGI", "CGI_shore", "non-CGI"]

pretty_feature_map = {
    "CGI": "CGI",
    "CGI_shore": "CGI shore",
    "non-CGI": "non-CGI",
}

sample_labels = {
    "DRG_0198_01": "DRG_0198_01",
    "DRG_0199_01": "DRG_0199_01",
    "DRG_0341_01": "DRG_0341_01",
    "DRG_1409_01": "DRG_1409_01",
    "DRG_1215_02": "DRG_1215_02",
    "DRG_2324_02": "DRG_2324_02",
    "DRG_2902_02": "DRG_2902_02",
    "DRG_5467_02": "DRG_5467_02",
}

preferred_sample_order = [
    "DRG_0341_01",
    "DRG_0198_01",
    "DRG_0199_01",
    "DRG_1409_01",
    "DRG_RG1215_02",
    "DRG_RG2324_02",
    "DRG_RG2902_02",
    "DRG_RG5467_02",
]

# No Unclassifiable
label_order = ["subject_to_XCI", "escapee"]

display_label_map = {
    "subject_to_XCI": "Inactive",
    "escapee": "escape",
}


def p_to_stars(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def add_sig_bar(ax, x1, x2, y, h, text):
    x1, x2 = sorted([x1, x2])

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=0.55,
        color="black",
        zorder=7,
        clip_on=False,
    )

    ax.text(
        (x1 + x2) / 2,
        y + h + 1.0,
        text,
        ha="center",
        va="bottom",
        fontsize=4.7,
        zorder=7,
        clip_on=False,
    )


def prepare_subset(
    df_gene_sample,
    feat,
    controls_only=False,
):
    sub = df_gene_sample[
        (df_gene_sample[feature_col] == feat)
        & df_gene_sample["label_overall"].isin(label_order)
    ].copy()

    sub = sub.dropna(
        subset=["methylation", "label_overall", "sample", "gene_symbol"]
    ).copy()

    sub["methylation"] = pd.to_numeric(sub["methylation"], errors="coerce")
    sub = sub.dropna(subset=["methylation"]).copy()

    if controls_only:
        sub = sub[
            ~sub["sample"].astype(str).str.startswith("RG")
        ].copy()

    sub["label_overall"] = pd.Categorical(
        sub["label_overall"],
        categories=label_order,
        ordered=True,
    )

    return sub


def get_sample_order(sub):
    present = set(sub["sample"].dropna().astype(str).unique())

    ordered = [
        s for s in preferred_sample_order
        if s in present
    ]

    remaining = sorted([
        s for s in present
        if s not in ordered
    ])

    return ordered + remaining


def run_donor_rlm_tests(sub, feat, sample_order, sample_set):
    rows = []

    term_escapee = (
        "C(label_overall, Treatment(reference='subject_to_XCI'))[T.escapee]"
    )

    for sample in sample_order:
        ss = sub[sub["sample"] == sample].copy()

        counts = ss["label_overall"].value_counts()

        n_inactive = int(counts.get("subject_to_XCI", 0))
        n_escape = int(counts.get("escapee", 0))

        row = {
            "feature": feat,
            "sample_set": sample_set,
            "sample": sample,
            "sample_label": sample_labels.get(sample, sample),
            "comparison": "escape vs Inactive",
            "n_inactive": n_inactive,
            "n_escape": n_escape,
            "n_gene_sample_rows": len(ss),
            "coef_escape_minus_inactive": np.nan,
            "std_err": np.nan,
            "z_value": np.nan,
            "p_value": np.nan,
        }

        # Need both groups present to run the model
        if n_inactive == 0 or n_escape == 0:
            rows.append(row)
            continue

        try:
            model = smf.rlm(
                "methylation ~ C(label_overall, Treatment(reference='subject_to_XCI'))",
                data=ss,
            ).fit()

            row["coef_escape_minus_inactive"] = model.params.get(term_escapee, np.nan)
            row["std_err"] = model.bse.get(term_escapee, np.nan)
            row["z_value"] = model.tvalues.get(term_escapee, np.nan)
            row["p_value"] = model.pvalues.get(term_escapee, np.nan)

        except Exception as e:
            row["error"] = str(e)

        rows.append(row)

    test_df = pd.DataFrame(rows)

    test_df["p_adj_fdr"] = np.nan

    ok = test_df["p_value"].notna()

    if ok.sum() > 0:
        test_df.loc[ok, "p_adj_fdr"] = multipletests(
            test_df.loc[ok, "p_value"],
            method="fdr_bh",
        )[1]

    test_df["stars_fdr"] = test_df["p_adj_fdr"].apply(p_to_stars)

    return test_df


def plot_methylation_dot_facets_by_donor(
    sub,
    feat,
    donor_tests,
    stem,
):
    sample_order = get_sample_order(sub)

    if len(sample_order) == 0:
        print(f"Skipping {stem}: no samples with data.")
        return

    n_samples = len(sample_order)

    # Force all donors into one row
    ncols = n_samples
    nrows = 1

    fig_w_mm = PANEL_WIDTH_MM * ncols
    fig_h_mm = PANEL_HEIGHT_MM

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(fig_w_mm / MM_PER_INCH, fig_h_mm / MM_PER_INCH),
        sharey=True,
    )

    if n_samples == 1:
        axes = np.array([axes])
    else:
        axes = np.array(axes).reshape(-1)

    rng = np.random.default_rng(1)

    positions = np.arange(1, len(label_order) + 1)
    display_labels = [display_label_map[x] for x in label_order]

    donor_tests_indexed = donor_tests.set_index("sample")

    for ax_i, sample in enumerate(sample_order):
        ax = axes[ax_i]

        sub_sample = sub[sub["sample"] == sample].copy()

        for x, lab in enumerate(label_order, start=1):
            vals = (
                sub_sample.loc[sub_sample["label_overall"] == lab, "methylation"]
                .dropna()
                .astype(float)
                .values
            )

            if len(vals) == 0:
                continue

            jitter = rng.uniform(-0.16, 0.16, size=len(vals))

            ax.scatter(
                np.full(len(vals), x) + jitter,
                vals,
                s=5.2,
                color="0.25",
                alpha=0.50,
                linewidths=0,
                zorder=3,
                rasterized=True,
            )

            # Median line for each class
            median_val = np.nanmedian(vals)

            ax.plot(
                [x - 0.22, x + 0.22],
                [median_val, median_val],
                color="black",
                linewidth=0.8,
                zorder=5,
                solid_capstyle="butt",
            )

        # FDR-corrected significance bar
        if sample in donor_tests_indexed.index:
            stars = donor_tests_indexed.loc[sample, "stars_fdr"]

            if pd.notna(stars) and stars != "NA":
                add_sig_bar(
                    ax=ax,
                    x1=1,
                    x2=2,
                    y=105,
                    h=3,
                    text=stars,
                )

        ax.set_title(
            sample_labels.get(sample, sample),
            pad=2.0,
            fontsize=5.8,
        )

        ax.set_xlim(0.55, len(label_order) + 0.45)
        ax.set_ylim(-3, 116)

        ax.set_xticks(positions)
        ax.set_xticklabels(
            display_labels,
            rotation=35,
            ha="right",
            rotation_mode="anchor",
        )

        ax.set_yticks([0, 20, 40, 60, 80, 100])

        ax.grid(axis="y", alpha=0.22, linewidth=0.4)
        ax.set_axisbelow(True)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # Avoid repeated y tick labels cluttering the row
        if ax_i != 0:
            ax.tick_params(labelleft=False)

        ax.tick_params(axis="x", pad=1.2)
        ax.tick_params(axis="y", pad=1.0)

    feature_label = pretty_feature_map.get(feat, feat)

    # Shared labels
    fig.text(
        0.50,
        0.045,
        "scDaisychain XCI classification",
        ha="center",
        va="center",
        fontsize=6.2,
    )

    fig.text(
        0.012,
        0.52,
        f"Percent methylated per\n{feature_label} promoter",
        ha="center",
        va="center",
        rotation="vertical",
        fontsize=6.2,
    )

    fig.patch.set_alpha(0)

    fig.subplots_adjust(
        left=0.055,
        right=0.995,
        bottom=0.30,
        top=0.82,
        wspace=0.35,
    )

    out_base = outdir / stem

    for ext in ["png", "pdf", "svg", "eps"]:
        fig.savefig(
            f"{out_base}.{ext}",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.04,
            transparent=True,
        )

    plt.show()

    print(f"Saved plot: {out_base}.png")
    print(f"Saved plot: {out_base}.pdf")
    print(f"Saved plot: {out_base}.svg")
    print(f"Saved plot: {out_base}.eps")


def make_feature_dot_facet_plots(
    df_gene_sample,
    controls_only=False,
):
    suffix = "controls_only" if controls_only else "all_samples"

    all_tests = []

    for feat in feature_list:
        print(f"\n=== {feat} | {suffix} | no_unclassifiable | dot facets by donor ===")

        sub = prepare_subset(
            df_gene_sample=df_gene_sample,
            feat=feat,
            controls_only=controls_only,
        )

        if sub.empty:
            print(
                f"Skipping {feat} {suffix}: "
                f"no data after filtering."
            )
            continue

        sample_order = get_sample_order(sub)

        print("Counts by donor and class:")
        print(
            sub.groupby(["sample", "label_overall"], observed=False)
            .size()
            .unstack(fill_value=0)
        )

        donor_tests = run_donor_rlm_tests(
            sub=sub,
            feat=feat,
            sample_order=sample_order,
            sample_set=suffix,
        )

        display(donor_tests)

        safe_feat = feat.replace("-", "_").replace("/", "_")

        n_samples = len(sample_order)
        fig_w_mm = PANEL_WIDTH_MM * n_samples
        fig_h_mm = PANEL_HEIGHT_MM

        stem = (
            f"{safe_feat}_methylation_by_XCI_class_"
            f"dotplot_one_row_faceted_by_donor_"
            f"no_unclassifiable_{suffix}_"
            f"RLM_FDR_"
            f"{fig_w_mm:.0f}x{fig_h_mm:.0f}mm"
        )

        plot_methylation_dot_facets_by_donor(
            sub=sub,
            feat=feat,
            donor_tests=donor_tests,
            stem=stem,
        )

        donor_tests["stem"] = stem
        all_tests.append(donor_tests)

        donor_tests.to_csv(
            outdir / f"{stem}_donor_RLM_FDR_tests.tsv",
            sep="\t",
            index=False,
        )

    return all_tests


# ============================================================
# Make plots
# ============================================================

all_donor_tests = []

all_donor_tests.extend(
    make_feature_dot_facet_plots(
        df_gene_sample=df_gene_sample,
        controls_only=False,
    )
)

all_donor_tests.extend(
    make_feature_dot_facet_plots(
        df_gene_sample=df_gene_sample,
        controls_only=True,
    )
)


# ============================================================
# Save combined donor-level test results
# ============================================================

if len(all_donor_tests) > 0:
    donor_RLM_FDR_tests_all_features = pd.concat(
        all_donor_tests,
        ignore_index=True,
    )

    donor_RLM_FDR_tests_all_features.to_csv(
        outdir / "methylation_dotplot_faceted_by_donor_RLM_FDR_tests_all_features.tsv",
        sep="\t",
        index=False,
    )

    display(donor_RLM_FDR_tests_all_features)
else:
    print("No donor-level RLM/FDR results to save.")

Each point is one gene-within-sample CGI methylation median.

The statistical test is a robust linear model:
    methylation ~ XCI_class + sample

This uses gene-level observations while adjusting for sample as a fixed effect.
Robust regression downweights outlying observations and is less sensitive to the non-normal, bounded methylation distribution than ordinary linear regression.

Pairwise contrasts between XCI classes are extracted from the fitted model, then FDR-corrected across the three comparisons.
Stars on the plot use the FDR-adjusted p-values.

In [ ]:
df_sample_summary = (
    df_gene_sample
    .groupby(["sample", "label_overall", feature_col], as_index=False)
    .agg(
        sample_median_methylation=("methylation", "median"),
        n_gene_sample=("methylation", "size"),
    )
)

df_sample_summary.head()

In [ ]:
from itertools import combinations
from scipy.stats import mannwhitneyu
import pandas as pd

p_rows = []

for feat in feature_order:
    sub = df_plot[df_plot[feature_col] == feat].copy()

    groups = {
        lab: sub.loc[sub["label_overall"] == lab, "fraction modified"]
               .dropna()
               .astype(float)
               .values
        for lab in label_order
    }

    for lab1, lab2 in combinations(label_order, 2):
        vals1 = groups[lab1]
        vals2 = groups[lab2]

        if len(vals1) == 0 or len(vals2) == 0:
            stat, p = pd.NA, pd.NA
        else:
            stat, p = mannwhitneyu(vals1, vals2, alternative="two-sided")

        p_rows.append({
            "CPG_feature": feat,
            "comparison": f"{lab1} vs {lab2}",
            "group1": lab1,
            "group2": lab2,
            "n_group1": len(vals1),
            "n_group2": len(vals2),
            "u_statistic": stat,
            "p_value": p,
            "stars": p_to_stars(p) if pd.notna(p) else pd.NA,
        })

pvals_df = pd.DataFrame(p_rows)
pvals_df

In [ ]:
df_sample_summary = (
    df_gene_sample
    .groupby(["sample", "label_overall", feature_col], as_index=False)
    .agg(
        sample_median_methylation=("methylation", "median"),
        n_gene_sample=("methylation", "size"),
    )
)

df_sample_summary.head()

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(feature_order),
    figsize=(2.8 * len(feature_order), 4.6),
    sharey=True,
)

if len(feature_order) == 1:
    axes = [axes]

for ax, feat in zip(axes, feature_order):
    sub = df_sample_summary[df_sample_summary[feature_col] == feat].copy()

    data = [
        sub.loc[sub["label_overall"] == lab, "sample_median_methylation"].dropna().astype(float).values
        for lab in label_order
    ]

    ax.boxplot(
        data,
        labels=display_labels,
        showfliers=False,
        patch_artist=False,
        boxprops=dict(color="black"),
        whiskerprops=dict(color="black"),
        capprops=dict(color="black"),
        medianprops=dict(color="black"),
    )

    # overlay sample points
    for i, lab in enumerate(label_order, start=1):
        vals = sub.loc[sub["label_overall"] == lab, "sample_median_methylation"].dropna().astype(float).values
        x = np.full(len(vals), i)
        ax.plot(x, vals, "o", color="black", markersize=3, alpha=0.7)

    ax.set_title(feat)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)

axes[0].set_ylabel("Sample median percent modified")

fig.suptitle("Sample-level methylation by XCI class")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
from itertools import combinations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

label_order = ["escapee", "subject_to_XCI", "uncertain"]

label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}

colors = {
    "escapee": "#1b9e77",
    "subject_to_XCI": "#d95f02",
    "uncertain": "#7570b3",
}

def p_to_stars(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def add_sig_bar(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, color="black")
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=7, color="black")

# ---------- within-sample pairwise tests ----------
p_rows = []

for feat in feature_order:
    for sample in sorted(df_gene_sample["sample"].dropna().unique()):
        sub = df_gene_sample[
            (df_gene_sample[feature_col] == feat)
            & (df_gene_sample["sample"] == sample)
        ].copy()

        groups = {
            lab: sub.loc[sub["label_overall"] == lab, "methylation"]
                   .dropna()
                   .astype(float)
                   .values
            for lab in label_order
        }

        for lab1, lab2 in combinations(label_order, 2):
            vals1 = groups[lab1]
            vals2 = groups[lab2]

            if len(vals1) == 0 or len(vals2) == 0:
                stat, p = pd.NA, pd.NA
            else:
                stat, p = mannwhitneyu(vals1, vals2, alternative="two-sided")

            p_rows.append({
                "CPG_feature": feat,
                "sample": sample,
                "comparison": f"{lab1} vs {lab2}",
                "group1": lab1,
                "group2": lab2,
                "n_group1_genes": sub.loc[sub["label_overall"] == lab1, "gene_symbol"].nunique(),
                "n_group2_genes": sub.loc[sub["label_overall"] == lab2, "gene_symbol"].nunique(),
                "u_statistic": stat,
                "p_value": p,
                "stars": p_to_stars(p),
            })

pvals_within_sample_df = pd.DataFrame(p_rows)

# ---------- faceted gene-level plots grouped by sample ----------
samples_order = sorted(df_gene_sample["sample"].dropna().unique())

fig, axes = plt.subplots(
    nrows=len(feature_order),
    ncols=len(samples_order),
    figsize=(3.2 * len(samples_order), 4.0 * len(feature_order)),
    sharey="row",
)

if len(feature_order) == 1 and len(samples_order) == 1:
    axes = np.array([[axes]])
elif len(feature_order) == 1:
    axes = np.array([axes])
elif len(samples_order) == 1:
    axes = np.array([[ax] for ax in axes])

for i, feat in enumerate(feature_order):
    for j, sample in enumerate(samples_order):
        ax = axes[i, j]

        sub = df_gene_sample[
            (df_gene_sample[feature_col] == feat)
            & (df_gene_sample["sample"] == sample)
        ].copy()

        data = [
            sub.loc[sub["label_overall"] == lab, "methylation"]
               .dropna()
               .astype(float)
               .values
            for lab in label_order
        ]

        if sum(len(x) for x in data) == 0:
            ax.set_axis_off()
            continue

        n_by_label = {
            lab: sub.loc[sub["label_overall"] == lab, "gene_symbol"].nunique()
            for lab in label_order
        }

        panel_labels = [
            f"{label_map[lab]}\n(n={n_by_label[lab]})"
            for lab in label_order
        ]

        bp = ax.boxplot(
            data,
            labels=panel_labels,
            showfliers=False,
            patch_artist=True,
            boxprops=dict(color="black"),
            whiskerprops=dict(color="black"),
            capprops=dict(color="black"),
            medianprops=dict(color="black", linewidth=1.5),
        )

        for patch, lab in zip(bp["boxes"], label_order):
            patch.set_facecolor(colors[lab])
            patch.set_alpha(0.75)

        for x, (lab, vals) in enumerate(zip(label_order, data), start=1):
            if len(vals) == 0:
                continue

            jitter = np.random.normal(0, 0.045, size=len(vals))

            ax.scatter(
                np.full(len(vals), x) + jitter,
                vals,
                s=7,
                color=colors[lab],
                alpha=0.35,
                linewidths=0,
            )

        panel_pvals = pvals_within_sample_df[
            (pvals_within_sample_df["CPG_feature"] == feat)
            & (pvals_within_sample_df["sample"] == sample)
        ]

        nonempty = [d for d in data if len(d) > 0]
        y_max = max(np.nanmax(d) for d in nonempty)
        y_min = min(np.nanmin(d) for d in nonempty)
        step = (y_max - y_min) * 0.12 if y_max > y_min else 0.05
        h = step * 0.3
        y = y_max + step

        for _, row in panel_pvals.iterrows():
            if pd.isna(row["p_value"]):
                continue

            x1 = label_order.index(row["group1"]) + 1
            x2 = label_order.index(row["group2"]) + 1

            add_sig_bar(ax, x1, x2, y, h, row["stars"])
            y += step

        ax.set_ylim(top=y + step)

        if i == 0:
            ax.set_title(sample, fontsize=10)

        if j == 0:
            ax.set_ylabel(f"{feat}\nPercent modified")
        else:
            ax.set_ylabel("")

        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45, labelsize=8)

fig.suptitle(
    "Gene-level methylation by sample and XCI status\nWithin-sample pairwise Mann–Whitney tests",
    y=1.02,
)

plt.tight_layout()
plt.show()

pvals_within_sample_df

In [ ]:

base = Path("/home/913/dk4874/scratch/gdata/scDaisychain_paper/human/processed_data")
bedmethyl_base = base / "bedmethyl"
top_tx_dir = Path("methylation_top_transcript_merged")
escapee_base = Path("/home/913/dk4874/scratch/forRob_daisychain/output/escapee_classification")

outdir = Path("plots")
outdir.mkdir(exist_ok=True)

win_size = 250
half_span = 5000
win_starts_rel = np.arange(-half_span, half_span, win_size)
win_centers_rel = win_starts_rel + win_size // 2

label_order = ["escapee", "subject_to_XCI", "uncertain"]
label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}

colors = {
    "escapee": "#1b9e77",
    "subject_to_XCI": "#d95f02",
    "uncertain": "#7570b3",
}

bed_cols = [
    "chrom",
    "start position",
    "end position",
    "modified base code",
    "score",
    "strand",
    "start position (compat)",
    "end position (compat)",
    "color",
    "Nvalid_cov",
    "fraction modified",
    "Nmod",
    "Ncanonical",
    "Nother_mod",
    "Ndelete",
    "Nfail",
    "Ndiff",
    "Nnocall",
]

def load_chrX_methyl(sample):
    bed_plain = (
        bedmethyl_base / sample /
        f"{sample}.reads.pass.modkit.coordA.cpg.chrX.bed"
    )

    bed_gz = (
        bedmethyl_base / sample /
        f"{sample}.reads.pass.modkit.coordA.cpg.sorted.bed.gz"
    )

    if bed_plain.exists():
        path = bed_plain
        df = pd.read_csv(
            path,
            sep="\t",
            header=None,
            names=bed_cols,
            usecols=range(18),
        )
    elif bed_gz.exists():
        path = bed_gz
        df = pd.read_csv(
            path,
            sep="\t",
            header=None,
            names=bed_cols,
            usecols=range(18),
            compression="gzip",
        )
        df = df[df["chrom"].astype(str).str.upper().eq("CHRX")].copy()
    else:
        raise FileNotFoundError(f"No chrX methyl BED found for {sample}")

    df = df[df["chrom"].astype(str).str.upper().eq("CHRX")].copy()
    df = df[df["modified base code"].astype(str).str.lower().eq("m")].copy()

    df["start position"] = pd.to_numeric(df["start position"], errors="coerce")
    df["fraction modified"] = pd.to_numeric(df["fraction modified"], errors="coerce")

    df = df.dropna(subset=["start position", "fraction modified"]).copy()
    df["start position"] = df["start position"].astype(int)

    return df.sort_values("start position").reset_index(drop=True)
def load_chrX_methyl_with_feature(sample):
    raw = load_chrX_methyl(sample)

    annot_path = (
        bedmethyl_base / sample /
        f"{sample}.reads.pass.modkit.coordA.cpg.chrX_annotated_with_CGI_shores_TSS.bed"
    )

    annot = pd.read_csv(
        annot_path,
        sep="\t",
        header=None,
        usecols=[0, 1, 2, 3, 18],
        names=["chrom", "start position", "end position", "modified base code", "CPG_feature"],
    )

    annot = annot[
        annot["chrom"].astype(str).str.upper().eq("CHRX")
        & annot["modified base code"].astype(str).str.lower().eq("m")
    ].copy()

    annot["start position"] = pd.to_numeric(annot["start position"], errors="coerce")
    annot = annot.dropna(subset=["start position"]).copy()
    annot["start position"] = annot["start position"].astype(int)

    annot = annot.drop_duplicates(["chrom", "start position", "modified base code"])

    raw = raw.merge(
        annot[["chrom", "start position", "modified base code", "CPG_feature"]],
        on=["chrom", "start position", "modified base code"],
        how="left",
    )

    raw["CPG_feature"] = raw["CPG_feature"].fillna("non-CGI")
    return raw.sort_values("start position").reset_index(drop=True)


def load_targets(sample):
    top_tx_path = top_tx_dir / f"{sample}_most_expressed_transcript_per_gene.tsv"
    escapee_path = escapee_base / f"PBMC_sample-{sample}_escapee_overall.csv"

    top_tx = pd.read_csv(top_tx_path, sep="\t")
    escapee = pd.read_csv(escapee_path)

    targets = top_tx.merge(
        escapee,
        how="left",
        left_on="gene_symbol",
        right_on="gene",
        suffixes=("", "_escapee"),
    )

    targets = targets[targets["label_overall"].isin(label_order)].copy()

    targets["tss"] = np.where(
        targets["tx_strand"].astype(str).eq("-"),
        pd.to_numeric(targets["tx_end"], errors="coerce") - 1,
        pd.to_numeric(targets["tx_start"], errors="coerce") - 1,
    )

    targets = targets.dropna(subset=["tss"]).copy()
    targets["tss"] = targets["tss"].astype(int)

    targets = targets[
        targets["chrom_tx"].astype(str).str.upper().eq("CHRX")
    ].copy()

    targets = targets[
        ["gene_symbol", "transcript_id", "label_overall", "chrom_tx", "tss", "tx_strand"]
    ].drop_duplicates()

    return targets

def window_profile_for_sample(sample):
    meth = load_chrX_methyl_with_feature(sample)
    targets = load_targets(sample)

    records = []

    starts = meth["start position"].values
    meth_vals = meth["fraction modified"].values
    features = meth["CPG_feature"].astype(str).values

    for _, row in targets.iterrows():
        gene = row["gene_symbol"]
        label = row["label_overall"]
        tss = int(row["tss"])

        region_start = max(0, tss - half_span)
        region_end = tss + half_span

        left = np.searchsorted(starts, region_start, side="left")
        right = np.searchsorted(starts, region_end, side="left")

        sub_starts = starts[left:right]
        sub_meth = meth_vals[left:right]
        sub_features = features[left:right]

        if len(sub_starts) == 0:
            continue

        rel_pos = sub_starts - tss
        keep = (rel_pos >= -half_span) & (rel_pos < half_span)

        rel_pos = rel_pos[keep]
        sub_meth = sub_meth[keep]
        sub_features = sub_features[keep]

        if len(rel_pos) == 0:
            continue

        bins = ((rel_pos + half_span) // win_size).astype(int)

        tmp = pd.DataFrame({
            "sample": sample,
            "gene_symbol": gene,
            "label_overall": label,
            "bin": bins,
            "fraction_modified": sub_meth,
            "CPG_feature": sub_features,
        })

        gene_bins = (
            tmp.groupby(
                ["sample", "gene_symbol", "label_overall", "CPG_feature", "bin"],
                as_index=False,
            )
            .agg(
                gene_window_methylation=("fraction_modified", "mean"),
                n_cpgs=("fraction_modified", "size"),
            )
        )

        gene_bins["rel_center"] = win_centers_rel[gene_bins["bin"].values]
        records.append(gene_bins)

    if not records:
        return pd.DataFrame()

    return pd.concat(records, ignore_index=True)


all_gene_windows = []

for sample in samples:
    print(f"Processing {sample}")
    prof = window_profile_for_sample(sample)
    print(sample, prof.shape)
    all_gene_windows.append(prof)

gene_window_all = pd.concat(all_gene_windows, ignore_index=True)

gene_window_all.to_csv(
    "all_samples_TSS_5kb_gene_window_methylation.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip",
)

sample_profile = (
    gene_window_all
    .groupby(["sample", "label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("gene_window_methylation", "mean"),
        n_genes=("gene_symbol", "nunique"),
    )
)

mean_profile = (
    sample_profile
    .groupby(["label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("mean_methylation", "mean"),
        sem_methylation=("mean_methylation", lambda x: x.std(ddof=1) / np.sqrt(x.notna().sum())),
        n_samples=("sample", "nunique"),
    )
)
fig, ax = plt.subplots(figsize=(6, 3.5))

for lab in label_order:
    sub = mean_profile[mean_profile["label_overall"] == lab].sort_values("rel_center")

    ax.plot(
        sub["rel_center"],
        sub["mean_methylation"],
        label=label_map[lab],
        linewidth=2,
        marker="o",
        markersize=3,
        color=colors[lab],
    )

    ax.fill_between(
        sub["rel_center"],
        sub["mean_methylation"] - sub["sem_methylation"],
        sub["mean_methylation"] + sub["sem_methylation"],
        color=colors[lab],
        alpha=0.2,
        linewidth=0,
    )

ax.axvline(0, linestyle="--", color="gray", linewidth=1.0, alpha=0.6)
ax.set_xlim(-half_span, half_span)
ax.set_xticks(np.arange(-half_span, half_span + 1, 1000))
ax.set_ylim(0, 100)
ax.set_xlabel("Distance from TSS (bp)")
ax.set_ylabel("Mean methylation (%)")
ax.set_title("Methylation in 250 bp windows ±5 kb from TSS")
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("plots/all_samples_TSS_5kb_methyl_escapee_vs_subject_to_XCI.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# TSS methylation profile by scDaisychain class
#
# Shading = ±1 SD BETWEEN SAMPLES
#
# Compact 60 mm × 50 mm version
# Font editable in Illustrator:
#   pdf.fonttype = 42
#   ps.fonttype = 42
#   svg.fonttype = "none"
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
samples=['DRG_0199_01','DRG_0198_01','DRG_0199_01','DRG_1215_02','DRG_2324_02','DRG_2902_02','DRG_5467_02','DRG_1409_01']

base = Path("/home/913/dk4874/scratch/gdata/scDaisychain_paper/human/processed_data")
bedmethyl_base = base / "bedmethyl"
top_tx_dir = Path("methylation_top_transcript_merged")
escapee_base = Path("/home/913/dk4874/scratch/forRob_daisychain/output/escapee_classification")

outdir = Path("plots")
outdir.mkdir(exist_ok=True)

# ---------- figure size ----------
MM_PER_INCH = 25.4
FIG_WIDTH_MM = 60
FIG_HEIGHT_MM = 50
FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "axes.titlesize": 6.2,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,
    "legend.fontsize": 4.8,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

# ---------- window settings ----------
win_size = 250
half_span = 5000

win_starts_rel = np.arange(-half_span, half_span, win_size)
win_centers_rel = win_starts_rel + win_size // 2

label_order = ["escapee", "subject_to_XCI", "uncertain"]

label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}

colors = {
    "escapee": "#1b9e77",
    "subject_to_XCI": "#d95f02",
    "uncertain": "#7570b3",
}

bed_cols = [
    "chrom",
    "start position",
    "end position",
    "modified base code",
    "score",
    "strand",
    "start position (compat)",
    "end position (compat)",
    "color",
    "Nvalid_cov",
    "fraction modified",
    "Nmod",
    "Ncanonical",
    "Nother_mod",
    "Ndelete",
    "Nfail",
    "Ndiff",
    "Nnocall",
]


def load_chrX_methyl(sample):
    bed_plain = (
        bedmethyl_base / sample /
        f"{sample}.reads.pass.modkit.coordA.cpg.chrX.bed"
    )

    bed_gz = (
        bedmethyl_base / sample /
        f"{sample}.reads.pass.modkit.coordA.cpg.sorted.bed.gz"
    )

    if bed_plain.exists():
        path = bed_plain
        df = pd.read_csv(
            path,
            sep="\t",
            header=None,
            names=bed_cols,
            usecols=range(18),
        )

    elif bed_gz.exists():
        path = bed_gz
        df = pd.read_csv(
            path,
            sep="\t",
            header=None,
            names=bed_cols,
            usecols=range(18),
            compression="gzip",
        )
        df = df[df["chrom"].astype(str).str.upper().eq("CHRX")].copy()

    else:
        raise FileNotFoundError(f"No chrX methyl BED found for {sample}")

    df = df[df["chrom"].astype(str).str.upper().eq("CHRX")].copy()
    df = df[df["modified base code"].astype(str).str.lower().eq("m")].copy()

    df["start position"] = pd.to_numeric(df["start position"], errors="coerce")
    df["fraction modified"] = pd.to_numeric(df["fraction modified"], errors="coerce")

    df = df.dropna(subset=["start position", "fraction modified"]).copy()
    df["start position"] = df["start position"].astype(int)

    return df.sort_values("start position").reset_index(drop=True)


def load_chrX_methyl_with_feature(sample):
    raw = load_chrX_methyl(sample)

    annot_path = (
        bedmethyl_base / sample /
        f"{sample}.reads.pass.modkit.coordA.cpg.chrX_annotated_with_CGI_shores_TSS.bed"
    )

    annot = pd.read_csv(
        annot_path,
        sep="\t",
        header=None,
        usecols=[0, 1, 2, 3, 18],
        names=[
            "chrom",
            "start position",
            "end position",
            "modified base code",
            "CPG_feature",
        ],
    )

    annot = annot[
        annot["chrom"].astype(str).str.upper().eq("CHRX")
        & annot["modified base code"].astype(str).str.lower().eq("m")
    ].copy()

    annot["start position"] = pd.to_numeric(
        annot["start position"],
        errors="coerce",
    )

    annot = annot.dropna(subset=["start position"]).copy()
    annot["start position"] = annot["start position"].astype(int)

    annot = annot.drop_duplicates(
        ["chrom", "start position", "modified base code"]
    )

    raw = raw.merge(
        annot[["chrom", "start position", "modified base code", "CPG_feature"]],
        on=["chrom", "start position", "modified base code"],
        how="left",
    )

    raw["CPG_feature"] = raw["CPG_feature"].fillna("non-CGI")

    return raw.sort_values("start position").reset_index(drop=True)


def load_targets(sample):
    top_tx_path = top_tx_dir / f"{sample}_most_expressed_transcript_per_gene.tsv"
    escapee_path = escapee_base / f"PBMC_sample-{sample}_escapee_overall.csv"

    top_tx = pd.read_csv(top_tx_path, sep="\t")
    escapee = pd.read_csv(escapee_path)

    targets = top_tx.merge(
        escapee,
        how="left",
        left_on="gene_symbol",
        right_on="gene",
        suffixes=("", "_escapee"),
    )

    targets = targets[targets["label_overall"].isin(label_order)].copy()

    targets["tss"] = np.where(
        targets["tx_strand"].astype(str).eq("-"),
        pd.to_numeric(targets["tx_end"], errors="coerce") - 1,
        pd.to_numeric(targets["tx_start"], errors="coerce") - 1,
    )

    targets = targets.dropna(subset=["tss"]).copy()
    targets["tss"] = targets["tss"].astype(int)

    targets = targets[
        targets["chrom_tx"].astype(str).str.upper().eq("CHRX")
    ].copy()

    targets = targets[
        [
            "gene_symbol",
            "transcript_id",
            "label_overall",
            "chrom_tx",
            "tss",
            "tx_strand",
        ]
    ].drop_duplicates()

    return targets


def window_profile_for_sample(sample):
    meth = load_chrX_methyl_with_feature(sample)
    targets = load_targets(sample)

    records = []

    starts = meth["start position"].values
    meth_vals = meth["fraction modified"].values
    features = meth["CPG_feature"].astype(str).values

    for _, row in targets.iterrows():
        gene = row["gene_symbol"]
        label = row["label_overall"]
        tss = int(row["tss"])

        region_start = max(0, tss - half_span)
        region_end = tss + half_span

        left = np.searchsorted(starts, region_start, side="left")
        right = np.searchsorted(starts, region_end, side="left")

        sub_starts = starts[left:right]
        sub_meth = meth_vals[left:right]
        sub_features = features[left:right]

        if len(sub_starts) == 0:
            continue

        rel_pos = sub_starts - tss

        keep = (rel_pos >= -half_span) & (rel_pos < half_span)

        rel_pos = rel_pos[keep]
        sub_meth = sub_meth[keep]
        sub_features = sub_features[keep]

        if len(rel_pos) == 0:
            continue

        bins = ((rel_pos + half_span) // win_size).astype(int)

        tmp = pd.DataFrame({
            "sample": sample,
            "gene_symbol": gene,
            "label_overall": label,
            "bin": bins,
            "fraction_modified": sub_meth,
            "CPG_feature": sub_features,
        })

        gene_bins = (
            tmp.groupby(
                ["sample", "gene_symbol", "label_overall", "CPG_feature", "bin"],
                as_index=False,
            )
            .agg(
                gene_window_methylation=("fraction_modified", "mean"),
                n_cpgs=("fraction_modified", "size"),
            )
        )

        gene_bins["rel_center"] = win_centers_rel[gene_bins["bin"].values]

        records.append(gene_bins)

    if not records:
        return pd.DataFrame()

    return pd.concat(records, ignore_index=True)


# ============================================================
# Build window methylation table
# ============================================================

all_gene_windows = []

for sample in samples:
    print(f"Processing {sample}")
    prof = window_profile_for_sample(sample)
    print(sample, prof.shape)
    all_gene_windows.append(prof)

gene_window_all = pd.concat(all_gene_windows, ignore_index=True)

gene_window_all.to_csv(
    "all_samples_TSS_5kb_gene_window_methylation.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip",
)


# ============================================================
# Sample-level profile first
#
# Each sample contributes one mean methylation value per class/window.
# Then the final shading is SD across those sample-level values.
# ============================================================

sample_profile = (
    gene_window_all
    .groupby(["sample", "label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("gene_window_methylation", "mean"),
        n_genes=("gene_symbol", "nunique"),
    )
)

mean_profile = (
    sample_profile
    .groupby(["label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("mean_methylation", "mean"),
        sd_between_samples=("mean_methylation", "std"),
        n_samples=("sample", "nunique"),
    )
)

mean_profile["sd_between_samples"] = mean_profile["sd_between_samples"].fillna(0)

mean_profile["lower_sd"] = (
    mean_profile["mean_methylation"] - mean_profile["sd_between_samples"]
).clip(lower=0)

mean_profile["upper_sd"] = (
    mean_profile["mean_methylation"] + mean_profile["sd_between_samples"]
).clip(upper=100)

mean_profile["rel_center_kb"] = mean_profile["rel_center"] / 1000


# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE)

for lab in label_order:
    sub = (
        mean_profile[mean_profile["label_overall"] == lab]
        .sort_values("rel_center")
        .copy()
    )

    if sub.empty:
        continue

    ax.plot(
        sub["rel_center_kb"],
        sub["mean_methylation"],
        label=label_map[lab],
        linewidth=1.0,
        marker="o",
        markersize=1.8,
        color=colors[lab],
    )

    ax.fill_between(
        sub["rel_center_kb"],
        sub["lower_sd"],
        sub["upper_sd"],
        color=colors[lab],
        alpha=0.18,
        linewidth=0,
    )

ax.axvline(
    0,
    linestyle="--",
    color="0.35",
    linewidth=0.65,
    alpha=0.7,
)

ax.set_xlim(-5, 5)
ax.set_xticks([-5, -2.5, 0, 2.5, 5])
ax.set_xticklabels(["-5", "-2.5", "0", "2.5", "5"])

ax.set_ylim(0, 100)
ax.set_yticks(np.arange(0, 101, 20))

ax.set_xlabel("Distance from TSS (kb)", labelpad=1.8)
ax.set_ylabel("Mean methylation (%)", labelpad=1.8)

# No title for compact panel

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(axis="y", alpha=0.22, linewidth=0.4)
ax.set_axisbelow(True)

ax.tick_params(axis="x", pad=1.5)
ax.tick_params(axis="y", pad=1.2)

ax.legend(
    frameon=False,
    loc="upper right",
    handlelength=1.2,
    handletextpad=0.4,
    labelspacing=0.25,
    borderaxespad=0.2,
    fontsize=4.8,
)

# ---------- transparent background ----------
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

fig.subplots_adjust(
    left=0.20,
    right=0.98,
    bottom=0.19,
    top=0.98,
)

out_base = outdir / "all_samples_TSS_5kb_methyl_escapee_vs_subject_to_XCI_SD_between_samples_60x50mm"

for ext in ["png", "pdf", "svg", "eps"]:
    fig.savefig(
        f"{out_base}.{ext}",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.03,
        transparent=True,
    )

plt.show()

In [ ]:
# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE)

for lab in label_order:
    sub = (
        mean_profile[mean_profile["label_overall"] == lab]
        .sort_values("rel_center")
        .copy()
    )

    if sub.empty:
        continue

    ax.plot(
        sub["rel_center_kb"],
        sub["mean_methylation"],
        label=label_map[lab],
        linewidth=1.0,
        marker="o",
        markersize=1.8,
        color=colors[lab],
    )

    ax.fill_between(
        sub["rel_center_kb"],
        sub["lower_sd"],
        sub["upper_sd"],
        color=colors[lab],
        alpha=0.18,
        linewidth=0,
    )

ax.axvline(
    0,
    linestyle="--",
    color="0.35",
    linewidth=0.65,
    alpha=0.7,
)

ax.set_xlim(-5, 5)
ax.set_xticks([-5, -2.5, 0, 2.5, 5])
ax.set_xticklabels(["-5", "-2.5", "0", "2.5", "5"])

ax.set_ylim(0, 100)
ax.set_yticks(np.arange(0, 101, 20))

ax.set_xlabel("Distance from TSS (kb)", labelpad=1.8)
ax.set_ylabel("Mean methylation (%)", labelpad=1.8)

# No title for compact panel

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(axis="y", alpha=0.22, linewidth=0.4)
ax.set_axisbelow(True)

ax.tick_params(axis="x", pad=1.5)
ax.tick_params(axis="y", pad=1.2)

# Legend centred at top
ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    handlelength=1.1,
    handletextpad=0.35,
    columnspacing=0.7,
    labelspacing=0.2,
    borderaxespad=0.0,
    fontsize=4.8,
)

# ---------- transparent background ----------
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

fig.subplots_adjust(
    left=0.20,
    right=0.98,
    bottom=0.19,
    top=0.90,
)

out_base = outdir / "all_samples_TSS_5kb_methyl_escapee_vs_subject_to_XCI_SD_between_samples_60x50mm"

for ext in ["png", "pdf", "svg", "eps"]:
    fig.savefig(
        f"{out_base}.{ext}",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.03,
        transparent=True,
    )

plt.show()

In [ ]:
# ---------- rebuild profiles with escapee + subject_to_XCI + uncertain ----------
# continuing from your variables/functions above

label_order = ["escapee", "subject_to_XCI", "uncertain"]

label_map = {
    "escapee": "Escapee",
    "subject_to_XCI": "Subject to XCI",
    "uncertain": "Unclassifiable",
}

colors = {
    "escapee": "#1b9e77",
    "subject_to_XCI": "#d95f02",
    "uncertain": "#7570b3",
}

all_gene_windows = []

for sample in samples:
    print(f"Processing {sample}")
    prof = window_profile_for_sample(sample)
    print(sample, prof.shape)
    all_gene_windows.append(prof)

gene_window_all = pd.concat(all_gene_windows, ignore_index=True)

print(gene_window_all["label_overall"].value_counts(dropna=False))

gene_window_all.to_csv(
    "all_samples_TSS_5kb_gene_window_methylation_3classes.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip",
)

sample_profile = (
    gene_window_all
    .groupby(["sample", "label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("gene_window_methylation", "mean"),
        sem_methylation=("gene_window_methylation", lambda x: x.std(ddof=1) / np.sqrt(x.notna().sum())),
        n_genes=("gene_symbol", "nunique"),
    )
)

mean_profile = (
    sample_profile
    .groupby(["label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("mean_methylation", "mean"),
        sem_methylation=("mean_methylation", lambda x: x.std(ddof=1) / np.sqrt(x.notna().sum())),
        n_samples=("sample", "nunique"),
    )
)

fig, ax = plt.subplots(figsize=(6.6, 4.0))

for lab in label_order:
    sub = mean_profile[
        mean_profile["label_overall"] == lab
    ].sort_values("rel_center")

    if sub.empty:
        print("No data for:", lab)
        continue

    ax.plot(
        sub["rel_center"],
        sub["mean_methylation"],
        label=label_map[lab],
        linewidth=2,
        marker="o",
        markersize=3,
        color=colors[lab],
    )

    ax.fill_between(
        sub["rel_center"],
        sub["mean_methylation"] - sub["sem_methylation"],
        sub["mean_methylation"] + sub["sem_methylation"],
        color=colors[lab],
        alpha=0.18,
        linewidth=0,
    )

ax.axvline(0, linestyle="--", color="gray", linewidth=1.0, alpha=0.6)
ax.set_xlim(-half_span, half_span)
ax.set_xticks(np.arange(-half_span, half_span + 1, 1000))
ax.set_ylim(0, 100)
ax.set_xlabel("Distance from TSS (bp)")
ax.set_ylabel("Mean methylation (%)")

ax.set_title(
    "Methylation in 250 bp windows ±5 kb from TSS",
    pad=14,
)

ax.legend(
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)

plt.tight_layout()
plt.savefig(
    "plots/all_samples_TSS_5kb_methyl_3classes.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# requires gene_window_all with CPG_feature column;
# use this only if you rebuilt with the CPG_feature-aware version

gene_window_cgi_promoters = gene_window_all[
    gene_window_all["CPG_feature"].isin(["CGI", "CGI_shore"])
    & gene_window_all["label_overall"].isin(label_order)
].copy()

sample_profile_cgi_promoters = (
    gene_window_cgi_promoters
    .groupby(["sample", "label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("gene_window_methylation", "mean"),
        sem_methylation=("gene_window_methylation", lambda x: x.std(ddof=1) / np.sqrt(x.notna().sum())),
        n_genes=("gene_symbol", "nunique"),
    )
)

mean_profile_cgi_promoters = (
    sample_profile_cgi_promoters
    .groupby(["label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("mean_methylation", "mean"),
        sem_methylation=("mean_methylation", lambda x: x.std(ddof=1) / np.sqrt(x.notna().sum())),
        n_samples=("sample", "nunique"),
    )
)

fig, ax = plt.subplots(figsize=(7, 4.0))

for lab in label_order:
    sub = mean_profile_cgi_promoters[
        mean_profile_cgi_promoters["label_overall"] == lab
    ].sort_values("rel_center")

    if sub.empty:
        print("No data for:", lab)
        continue

    ax.plot(
        sub["rel_center"],
        sub["mean_methylation"],
        label=label_map[lab],
        linewidth=2,
        marker="o",
        markersize=3,
        color=colors[lab],
    )

    ax.fill_between(
        sub["rel_center"],
        sub["mean_methylation"] - sub["sem_methylation"],
        sub["mean_methylation"] + sub["sem_methylation"],
        color=colors[lab],
        alpha=0.18,
        linewidth=0,
    )

ax.axvline(0, linestyle="--", color="gray", linewidth=1.0, alpha=0.6)
ax.set_xlim(-half_span, half_span)
ax.set_xticks(np.arange(-half_span, half_span + 1, 1000))
ax.set_ylim(0, 100)
ax.set_xlabel("Distance from TSS (bp)")
ax.set_ylabel("Mean methylation (%)")

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
)

ax.set_title(
    "Average TSS methylation profile at CGI/CGI-shore promoters",
    pad=10,
)

plt.tight_layout()




out_base = "plots/average_TSS_5kb_methyl_CGI_CGIshore_only_3classes"

plt.savefig(f"{out_base}.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")
plt.savefig(f"{out_base}.svg", bbox_inches="tight")
plt.savefig(f"{out_base}.eps", bbox_inches="tight")  # assuming you meant EPS
plt.show()

In [ ]:
sample_profile_cgi_promoters = (
    gene_window_all[
        gene_window_all["CPG_feature"].isin(["CGI", "CGI_shore"])
        & gene_window_all["label_overall"].isin(label_order)
    ]
    .groupby(["sample", "label_overall", "rel_center"], as_index=False)
    .agg(
        mean_methylation=("gene_window_methylation", "mean"),
        sem_methylation=("gene_window_methylation", lambda x: x.std(ddof=1) / np.sqrt(x.notna().sum())),
        n_genes=("gene_symbol", "nunique"),
    )
)

samples_order = sorted(sample_profile_cgi_promoters["sample"].dropna().unique())

fig, axes = plt.subplots(
    nrows=1,
    ncols=len(samples_order),
    figsize=(3.2 * len(samples_order), 3.4),
    sharey=True,
)

if len(samples_order) == 1:
    axes = [axes]

for ax, sample in zip(axes, samples_order):
    for lab in label_order:
        sub = sample_profile_cgi_promoters[
            (sample_profile_cgi_promoters["sample"] == sample)
            & (sample_profile_cgi_promoters["label_overall"] == lab)
        ].sort_values("rel_center")

        if sub.empty:
            continue

        ax.plot(
            sub["rel_center"],
            sub["mean_methylation"],
            label=label_map[lab],
            linewidth=2,
            marker="o",
            markersize=3,
            color=colors[lab],
        )

        ax.fill_between(
            sub["rel_center"],
            sub["mean_methylation"] - sub["sem_methylation"],
            sub["mean_methylation"] + sub["sem_methylation"],
            color=colors[lab],
            alpha=0.18,
            linewidth=0,
        )

    ax.axvline(0, linestyle="--", color="gray", linewidth=1.0, alpha=0.6)
    ax.set_xlim(-half_span, half_span)
    ax.set_xticks([-5000, -2500, 0, 2500, 5000])
    ax.set_ylim(0, 100)
    ax.set_title(sample)
    ax.set_xlabel("Distance from TSS (bp)")

axes[0].set_ylabel("Mean methylation (%)")
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.96),
    ncol=3,
    frameon=False,
)

fig.suptitle(
    "TSS methylation profiles by sample at CGI/CGI-shore promoters",
    y=1.02,
)

plt.tight_layout(rect=[0, 0, 1, 0.98])



out_base = "plots/per_sample_TSS_5kb_methyl_CGI_CGIshore_only_3classes_with_SEM"

plt.savefig(f"{out_base}.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")
plt.savefig(f"{out_base}.svg", bbox_inches="tight")
plt.savefig(f"{out_base}.eps", bbox_inches="tight")  # assuming you meant EPS
plt.show()